In [42]:
"""
Enhanced Personalized Interview System - Functional Version
All classes converted to functions with data structures
Keeping ALL print statements intact
"""

from typing import List, Dict, Set, Optional, Tuple
from datetime import datetime
from dataclasses import dataclass, field
from collections import defaultdict
import json

In [43]:
# ==================== DATA STRUCTURES ====================
@dataclass
class UserProfile:
    """User Profile Data Structure"""
    _id: str
    name: str
    email: str
    skills: List[str]
    skill_levels: Dict[str, int]
    years_experience: float
    education: str
    target_domains: List[str]
    target_companies: List[str]
    created_at: str
    last_active: str
    weaker_skills: List[str] = field(default_factory=list)
    stronger_skills: List[str] = field(default_factory=list)
    skill_metadata: Dict[str, Dict] = field(default_factory=dict)
    domain_specific_weaker_skills: Dict[str, List[str]] = field(default_factory=dict)
    domain_specific_stronger_skills: Dict[str, List[str]] = field(default_factory=dict)
    domain_specific_skill_metadata: Dict[str, Dict[str, Dict]] = field(default_factory=dict)


@dataclass
class Question:
    """Question Bank Data Structure"""
    _id: str
    text: str
    tags: List[str]
    domain: str
    difficulty: int
    estimated_time_sec: int
    embedding: List[float] = field(default_factory=list)
    created_by: str = "editor"
    popularity: int = 0
    company_specific: List[str] = field(default_factory=list)
    last_updated: str = ""


@dataclass
class QuestionResponse:
    """Individual Question Response with Enhanced Metadata"""
    _id: str
    session_id: str
    user_id: str
    question_id: str
    response_text: str
    response_audio_features: Dict[str, float]
    content_metrics: Dict[str, float]
    final_score: float
    timestamp: str
    time_taken_sec: int = 0
    skipped: bool = False
    confidence_rating: Optional[int] = None
    keyword_matches: List[str] = field(default_factory=list)
    domain: str = ""


@dataclass
class Session:
    """User Session Data with Enhanced Tracking"""
    _id: str
    user_id: str
    context: Dict[str, str]
    questions: List[Dict[str, str]]
    created_at: str
    metrics_snapshot: Dict[str, float]
    skill_performance: Dict[str, List[float]] = field(default_factory=dict)
    weaker_skills_identified: List[str] = field(default_factory=list)
    stronger_skills_identified: List[str] = field(default_factory=list)
    total_time_sec: int = 0
    questions_skipped: int = 0
    domain: str = ""


@dataclass
class SkillAnalysis:
    """Detailed skill analysis from session data"""
    skill: str
    avg_score: float
    total_questions: int
    avg_time_sec: float
    skip_rate: float
    avg_confidence: Optional[float]
    keyword_match_rate: float
    category: str


@dataclass
class RecommendedQuestion:
    """Recommended Question Output with Reasoning"""
    q_id: str
    score: float
    reasoning: str = ""


In [44]:
# ==================== GLOBAL STATE ====================

def initialize_system(question_bank: List[Question]) -> Dict:
    """Initialize system state"""
    print(f"\n{'='*70}")
    print(f"[INIT] Enhanced Interview System Initialized")
    print(f"[INIT] Total Questions in Bank: {len(question_bank)}")
    print(f"{'='*70}\n")
    
    return {
        'question_bank': question_bank,
        'sessions_db': defaultdict(list),
        'responses_db': defaultdict(list),
        'skill_vectors': {},
        'user_profiles': {},
        'domain_sessions_db': defaultdict(lambda: defaultdict(list)),
        'domain_responses_db': defaultdict(lambda: defaultdict(list))
    }

In [45]:
# ==================== HELPER FUNCTIONS ====================

def filter_by_domain(question_bank: List[Question], target_domains: List[str]) -> List[Question]:
    """Filter questions by user's target domains"""
    return [q for q in question_bank if q.domain in target_domains]


def calculate_skill_overlap(question_tags: List[str], user_skills: List[str]) -> float:
    """Calculate skill overlap score"""
    if not question_tags:
        return 0.0
    intersection = len(set(question_tags) & set(user_skills))
    return intersection / len(question_tags)


def get_last_score(state: Dict, user_id: str, domain: str = None) -> float:
    """Get the last response score"""
    if domain:
        responses = state['domain_responses_db'][user_id].get(domain, [])
    else:
        responses = state['responses_db'].get(user_id, [])
    
    if not responses:
        return 0.5
    sorted_responses = sorted(responses, key=lambda x: x.timestamp)
    return sorted_responses[-1].final_score


def calculate_difficulty_adjustment(last_score: float) -> int:
    """Adjust difficulty based on last score"""
    if last_score >= 0.7:
        return 1
    elif last_score < 0.4:
        return -1
    else:
        return 0


def get_attempted_questions(state: Dict, user_id: str, domain: str = None) -> Set[str]:
    """Get set of question IDs already attempted"""
    if domain:
        responses = state['domain_responses_db'][user_id].get(domain, [])
    else:
        responses = state['responses_db'].get(user_id, [])
    return {response.question_id for response in responses}


def filter_candidate_questions(question_bank: List[Question], target_domains: List[str],
                               skills: List[str], target_difficulty: int,
                               attempted_questions: Set[str]) -> List[Question]:
    """Filter questions based on criteria"""
    candidates = []
    for question in question_bank:
        if question.domain not in target_domains:
            continue
        if not set(question.tags) & set(skills):
            continue
        if abs(question.difficulty - target_difficulty) > 1:
            continue
        if question._id in attempted_questions:
            continue
        candidates.append(question)
    return candidates


def get_question_by_id(question_bank: List[Question], question_id: str) -> Optional[Question]:
    """Retrieve question by ID"""
    for q in question_bank:
        if q._id == question_id:
            return q
    return None


In [46]:
# ==================== DIVERSE ASSESSMENT ====================

def create_diverse_assessment_set(questions: List[Question], skills: List[str], k: int) -> List[Question]:
    """Create diverse question set ensuring coverage of all skills"""
    skill_coverage = defaultdict(list)
    
    # Group questions by skills they test
    for question in questions:
        for tag in question.tags:
            if tag in skills:
                skill_coverage[tag].append(question)
    
    print(f"      - Skill Coverage Analysis:")
    for skill, qs in skill_coverage.items():
        print(f"        {skill}: {len(qs)} questions available")
    
    # Select questions ensuring each skill is tested
    selected = []
    selected_ids = set()
    
    # Round-robin selection from each skill
    max_per_skill = max(1, k // len(skills))
    print(f"      - Target: {max_per_skill} questions per skill")
    
    for skill in skills:
        skill_questions = skill_coverage.get(skill, [])
        for q in skill_questions[:max_per_skill]:
            if q._id not in selected_ids:
                selected.append(q)
                selected_ids.add(q._id)
                if len(selected) >= k:
                    break
        if len(selected) >= k:
            break
    
    # Fill remaining slots if needed
    if len(selected) < k:
        for q in questions:
            if q._id not in selected_ids:
                selected.append(q)
                selected_ids.add(q._id)
                if len(selected) >= k:
                    break
    
    print(f"      ✓ Selected {len(selected)} diverse questions")
    return selected

In [47]:
# ==================== SKILL ANALYSIS ====================

def analyze_user_skills(state: Dict, user_id: str, domain: str = None) -> List[SkillAnalysis]:
    """Analyze user skills from response history"""
    if domain:
        responses = state['domain_responses_db'][user_id].get(domain, [])
    else:
        responses = state['responses_db'].get(user_id, [])
    
    if not responses:
        return []
    
    skill_data = defaultdict(lambda: {
        'scores': [],
        'times': [],
        'skips': [],
        'confidences': [],
        'keyword_matches': []
    })
    
    # Aggregate data per skill
    for response in responses:
        question = get_question_by_id(state['question_bank'], response.question_id)
        if not question:
            continue
        
        for tag in question.tags:
            skill_data[tag]['scores'].append(response.final_score)
            skill_data[tag]['times'].append(response.time_taken_sec)
            skill_data[tag]['skips'].append(1 if response.skipped else 0)
            if response.confidence_rating:
                skill_data[tag]['confidences'].append(response.confidence_rating)
            if response.keyword_matches:
                skill_data[tag]['keyword_matches'].append(len(response.keyword_matches))
    
    # Create analyses
    analyses = []
    for skill, data in skill_data.items():
        if not data['scores']:
            continue
        
        avg_score = sum(data['scores']) / len(data['scores'])
        avg_time = sum(data['times']) / len(data['times']) if data['times'] else 0
        skip_rate = sum(data['skips']) / len(data['skips'])
        avg_confidence = (
            sum(data['confidences']) / len(data['confidences'])
            if data['confidences'] else None
        )
        keyword_match_rate = (
            sum(data['keyword_matches']) / len(data['keyword_matches'])
            if data['keyword_matches'] else 0
        )
        
        # Categorize skill
        if avg_score < 0.5 or skip_rate > 0.3:
            category = 'weaker'
        elif avg_score > 0.7 and skip_rate < 0.1:
            category = 'stronger'
        else:
            category = 'moderate'
        
        analyses.append(SkillAnalysis(
            skill=skill,
            avg_score=avg_score,
            total_questions=len(data['scores']),
            avg_time_sec=avg_time,
            skip_rate=skip_rate,
            avg_confidence=avg_confidence,
            keyword_match_rate=keyword_match_rate,
            category=category
        ))
    
    return sorted(analyses, key=lambda x: x.avg_score)


def score_profile_based_questions(questions: List[Question], skill_analyses: List[SkillAnalysis],
                                  focus_type: str) -> List[Tuple[Question, float, str]]:
    """Score questions based on user profile analysis"""
    scored = []
    skill_map = {a.skill: a for a in skill_analyses}
    
    for question in questions:
        # Calculate metrics
        question_skills = [s for s in question.tags if s in skill_map]
        
        if not question_skills:
            continue
        
        # Get average score for question skills
        avg_skill_score = sum(
            skill_map[s].avg_score for s in question_skills
        ) / len(question_skills)
        
        # Get average time taken for these skills
        avg_time = sum(
            skill_map[s].avg_time_sec for s in question_skills
        ) / len(question_skills)
        
        # Calculate priority score
        if focus_type == 'weaker':
            # Lower score = higher priority for weaker skills
            priority = (1 - avg_skill_score)
            time_factor = min(1.0, avg_time / 180)
            score = priority + (0.3 * time_factor)
            
            reasoning = (
                f"Focus on {', '.join(question_skills)} "
                f"(accuracy: {avg_skill_score:.0%}, "
                f"avg time: {avg_time:.0f}s) - needs improvement"
            )
        else:
            # Higher score = validate mastery
            priority = avg_skill_score
            score = priority
            
            reasoning = (
                f"Validate {', '.join(question_skills)} "
                f"(strong performance: {avg_skill_score:.0%}) - maintain mastery"
            )
        
        scored.append((question, score, reasoning))
    
    return scored

In [48]:
# ==================== FIRST SESSION FLOW ====================

def first_session_recommendations(state: Dict, user: UserProfile, k: int, 
                                 chosen_domain: str = None) -> Dict:
    """First Session: Diverse question set to assess all skills"""
    print(f"\n[PHASE 1] First Session - Comprehensive Assessment")
    print(f"{'─'*70}")
    print(f"[1.1] Goal: Assess all user skills comprehensively")
    print(f"[1.2] User Skills to Assess: {user.skills}")
    
    # Filter by domain
    target_domains = [chosen_domain] if chosen_domain else user.target_domains
    print(f"\n[1.3] Filtering by domain: {target_domains}")
    domain_questions = filter_by_domain(state['question_bank'], target_domains)
    print(f"      ✓ {len(domain_questions)} questions in target domains")
    
    # Create diverse question set covering all skills
    print(f"\n[1.4] Creating diverse question set...")
    diverse_questions = create_diverse_assessment_set(domain_questions, user.skills, k)
    
    # Score and rank
    print(f"\n[1.5] Scoring questions for initial assessment...")
    scored_questions = []
    for i, question in enumerate(diverse_questions):
        print(f"\n      Question {i+1}/{len(diverse_questions)}: {question._id}")
        print(f"      - Tags: {question.tags}")
        print(f"      - Difficulty: {question.difficulty}")
        
        # Prioritize skill coverage over difficulty match
        skill_overlap = calculate_skill_overlap(question.tags, user.skills)
        
        # Slight penalty for extreme difficulties in first session
        difficulty_factor = 1.0
        if question.difficulty == 1 or question.difficulty == 5:
            difficulty_factor = 0.9
        
        score = skill_overlap * difficulty_factor
        print(f"      - Skill Overlap: {skill_overlap:.2f}")
        print(f"      - Difficulty Factor: {difficulty_factor:.2f}")
        print(f"      - Final Score: {score:.2f}")
        
        scored_questions.append((question, score))
    
    # Rank and select
    ranked = sorted(scored_questions, key=lambda x: x[1], reverse=True)
    
    recommendations = [
        RecommendedQuestion(
            q_id=q._id, 
            score=round(score, 2),
            reasoning=f"Initial assessment for {', '.join(q.tags)}"
        )
        for q, score in ranked[:k]
    ]
    
    print(f"\n[1.6] ✓ First Session Recommendations Ready")
    print(f"      - Total Recommended: {len(recommendations)}")
    for i, rec in enumerate(recommendations):
        print(f"      {i+1}. {rec.q_id} (Score: {rec.score}) - {rec.reasoning}")
    
    output = {
        "user_id": user._id,
        "session_type": "cold_start_first_session",
        "phase": "data_collection",
        "domain": chosen_domain if chosen_domain else user.target_domains[0],
        "message": "First session: Comprehensive skill assessment",
        "questions_recommended": [
            {"q_id": r.q_id, "score": r.score, "reasoning": r.reasoning}
            for r in recommendations
        ]
    }
    
    print(f"{'='*70}\n")
    return output


In [49]:
# ==================== PROFILE-BASED FLOW ====================

def profile_based_recommendations(state: Dict, user: UserProfile, k: int, 
                                 chosen_domain: str = None) -> Dict:
    """Subsequent Sessions: Use built profile for targeted recommendations"""
    print(f"\n[PHASE 2] Profile-Based Recommendations")
    print(f"{'─'*70}")
    
    # Analyze user profile for specific domain
    print(f"[2.1] Analyzing User Profile for Domain: {chosen_domain}...")
    skill_analyses = analyze_user_skills(state, user._id, chosen_domain)
    
    if not skill_analyses:
        print(f"      ⚠ No profile data found, falling back to first session flow")
        return first_session_recommendations(state, user, k, chosen_domain)
    
    print(f"\n[2.2] Skill Analysis Summary:")
    for analysis in skill_analyses:
        print(f"      - {analysis.skill}: {analysis.category.upper()}")
        print(f"        Avg Score: {analysis.avg_score:.2f}, "
              f"Questions: {analysis.total_questions}, "
              f"Avg Time: {analysis.avg_time_sec:.0f}s")
    
    # Identify weaker and stronger skills
    weaker_skills = [a.skill for a in skill_analyses if a.category == 'weaker']
    stronger_skills = [a.skill for a in skill_analyses if a.category == 'stronger']
    
    print(f"\n[2.3] Skill Categorization:")
    print(f"      - Weaker Skills (need improvement): {weaker_skills}")
    print(f"      - Stronger Skills (validation): {stronger_skills}")
    
    # Get adaptive difficulty for specific domain
    last_score = get_last_score(state, user._id, chosen_domain)
    difficulty_adjustment = calculate_difficulty_adjustment(last_score)
    target_difficulty = max(1, min(5, 2 + difficulty_adjustment))
    
    print(f"\n[2.4] Difficulty Adaptation:")
    print(f"      - Last Score: {last_score:.2f}")
    print(f"      - Adjustment: {difficulty_adjustment:+d}")
    print(f"      - Target Difficulty: {target_difficulty}")
    
    # Filter candidates
    attempted = get_attempted_questions(state, user._id, chosen_domain)
    
    print(f"\n[2.5] Filtering Candidate Questions...")
    print(f"      - Previously Attempted: {len(attempted)}")
    
    # Get questions focused on weaker skills (70%)
    target_domains = [chosen_domain] if chosen_domain else user.target_domains
    weaker_candidates = filter_candidate_questions(
        state['question_bank'], target_domains, weaker_skills, target_difficulty, attempted
    )
    
    # Get questions for stronger skills (30%)
    stronger_candidates = filter_candidate_questions(
        state['question_bank'], target_domains, stronger_skills, target_difficulty, attempted
    )
    
    print(f"      - Weaker Skills Candidates: {len(weaker_candidates)}")
    print(f"      - Stronger Skills Candidates: {len(stronger_candidates)}")
    
    # Score and rank
    print(f"\n[2.6] Scoring and Ranking Questions...")
    
    weaker_scored = score_profile_based_questions(weaker_candidates, skill_analyses, 'weaker')
    stronger_scored = score_profile_based_questions(stronger_candidates, skill_analyses, 'stronger')
    
    # Allocate 70% to weaker, 30% to stronger
    k_weaker = int(k * 0.7)
    k_stronger = k - k_weaker
    
    weaker_ranked = sorted(weaker_scored, key=lambda x: x[1], reverse=True)[:k_weaker]
    stronger_ranked = sorted(stronger_scored, key=lambda x: x[1], reverse=True)[:k_stronger]
    
    # Combine recommendations
    recommendations = []
    
    for q, score, reasoning in weaker_ranked:
        recommendations.append(RecommendedQuestion(
            q_id=q._id, score=round(score, 2), reasoning=reasoning
        ))
    
    for q, score, reasoning in stronger_ranked:
        recommendations.append(RecommendedQuestion(
            q_id=q._id, score=round(score, 2), reasoning=reasoning
        ))
    
    print(f"\n[2.7] ✓ Final Recommendations:")
    print(f"      - Focus on Weaker Skills: {k_weaker} questions")
    print(f"      - Validate Stronger Skills: {k_stronger} questions")
    
    for i, rec in enumerate(recommendations):
        print(f"      {i+1}. {rec.q_id} (Score: {rec.score})")
        print(f"         → {rec.reasoning}")
    
    output = {
        "user_id": user._id,
        "session_type": "cold_start_profile_based",
        "phase": "adaptive_learning",
        "domain": chosen_domain if chosen_domain else user.target_domains[0],
        "weaker_skills_targeted": weaker_skills,
        "stronger_skills_validated": stronger_skills,
        "skill_analyses": [
            {
                "skill": a.skill, "category": a.category,
                "avg_score": round(a.avg_score, 2),
                "total_questions": a.total_questions,
                "avg_time_sec": round(a.avg_time_sec, 1)
            }
            for a in skill_analyses
        ],
        "questions_recommended": [
            {"q_id": r.q_id, "score": r.score, "reasoning": r.reasoning}
            for r in recommendations
        ]
    }
    
    print(f"{'='*70}\n")
    return output

In [50]:
# ==================== ENHANCED COLD START ====================

def enhanced_cold_start_flow(state: Dict, user: UserProfile, k: int = 10, 
                             is_first_session: bool = True, chosen_domain: str = None) -> Dict:
    """Enhanced Cold Start Flow with Profile Building"""
    print(f"\n{'='*70}")
    print(f"[ENHANCED COLD START] Profile-Building Flow")
    print(f"{'='*70}")
    print(f"[0.1] User: {user._id}")
    print(f"[0.2] Is First Session: {is_first_session}")
    print(f"[0.3] Chosen Domain: {chosen_domain}")
    
    if is_first_session:
        return first_session_recommendations(state, user, k, chosen_domain)
    else:
        return profile_based_recommendations(state, user, k, chosen_domain)

In [51]:
# ==================== PROFILE BUILDING ====================

def build_user_profile_from_session(state: Dict, user_id: str, session_id: str, domain: str = None):
    """Build/Update user profile from completed session data"""
    print(f"\n{'='*70}")
    print(f"[PROFILE BUILDING] Building User Profile from Session")
    print(f"{'='*70}")
    print(f"[PB.1] User ID: {user_id}")
    print(f"[PB.2] Session ID: {session_id}")
    print(f"[PB.3] Domain: {domain}")
    
    # Get all responses for this session
    if domain:
        session_responses = [
            r for r in state['domain_responses_db'][user_id].get(domain, [])
            if r.session_id == session_id
        ]
    else:
        session_responses = [
            r for r in state['responses_db'].get(user_id, [])
            if r.session_id == session_id
        ]
    
    if not session_responses:
        print(f"[PB.4] ⚠ No responses found for session")
        return
    
    print(f"[PB.4] Processing {len(session_responses)} responses...")
    
    # Analyze skills
    skill_analyses = analyze_user_skills(state, user_id, domain)
    
    # Update user profile
    if user_id in state['user_profiles']:
        user_profile = state['user_profiles'][user_id]
    else:
        print(f"[PB.5] ⚠ User profile not found in cache")
        return
    
    # Update profile fields
    weaker_skills = [a.skill for a in skill_analyses if a.category == 'weaker']
    stronger_skills = [a.skill for a in skill_analyses if a.category == 'stronger']
    
    if domain:
        # Store domain-specific data
        user_profile.domain_specific_weaker_skills[domain] = weaker_skills
        user_profile.domain_specific_stronger_skills[domain] = stronger_skills
        
        # Store detailed skill metadata for domain
        if domain not in user_profile.domain_specific_skill_metadata:
            user_profile.domain_specific_skill_metadata[domain] = {}
        
        for analysis in skill_analyses:
            user_profile.domain_specific_skill_metadata[domain][analysis.skill] = {
                'avg_score': analysis.avg_score,
                'total_questions': analysis.total_questions,
                'avg_time_sec': analysis.avg_time_sec,
                'skip_rate': analysis.skip_rate,
                'category': analysis.category
            }
    else:
        # Store global data
        user_profile.weaker_skills = weaker_skills
        user_profile.stronger_skills = stronger_skills
        
        for analysis in skill_analyses:
            user_profile.skill_metadata[analysis.skill] = {
                'avg_score': analysis.avg_score,
                'total_questions': analysis.total_questions,
                'avg_time_sec': analysis.avg_time_sec,
                'skip_rate': analysis.skip_rate,
                'category': analysis.category
            }
    
    print(f"\n[PB.5] ✓ Profile Updated:")
    print(f"      - Weaker Skills: {weaker_skills}")
    print(f"      - Stronger Skills: {stronger_skills}")
    print(f"      - Total Skills Analyzed: {len(skill_analyses)}")
    
    print(f"\n[PB.6] Detailed Skill Metadata:")
    skill_metadata = (user_profile.domain_specific_skill_metadata.get(domain, {}) 
                     if domain else user_profile.skill_metadata)
    for skill, metadata in skill_metadata.items():
        print(f"      - {skill}:")
        print(f"        Category: {metadata['category']}")
        print(f"        Avg Score: {metadata['avg_score']:.2f}")
        print(f"        Questions Answered: {metadata['total_questions']}")
        print(f"        Avg Time: {metadata['avg_time_sec']:.0f}s")
        print(f"        Skip Rate: {metadata['skip_rate']:.0%}")
    
    print(f"{'='*70}\n")

In [52]:
# ==================== LOGGING FUNCTIONS ====================

def log_response(state: Dict, response: QuestionResponse):
    """Log user response with enhanced metadata"""
    print(f"\n{'─'*70}")
    print(f"[RESPONSE LOG] Logging Enhanced Response")
    print(f"{'─'*70}")
    print(f"- User ID: {response.user_id}")
    print(f"- Question ID: {response.question_id}")
    print(f"- Final Score: {response.final_score:.2f}")
    print(f"- Time Taken: {response.time_taken_sec}s")
    print(f"- Skipped: {response.skipped}")
    print(f"- Confidence: {response.confidence_rating}")
    print(f"- Keyword Matches: {response.keyword_matches}")
    print(f"- Domain: {response.domain}")
    
    # Store in both global and domain-specific databases
    state['responses_db'][response.user_id].append(response)
    if response.domain:
        state['domain_responses_db'][response.user_id][response.domain].append(response)
    
    print(f"✓ Response logged")
    print(f"{'─'*70}\n")


def log_session(state: Dict, session: Session):
    """Log session with enhanced tracking"""
    print(f"\n{'─'*70}")
    print(f"[SESSION LOG] Logging Enhanced Session")
    print(f"{'─'*70}")
    print(f"- Session ID: {session._id}")
    print(f"- User ID: {session.user_id}")
    print(f"- Total Time: {session.total_time_sec}s")
    print(f"- Questions Skipped: {session.questions_skipped}")
    print(f"- Weaker Skills: {session.weaker_skills_identified}")
    print(f"- Stronger Skills: {session.stronger_skills_identified}")
    print(f"- Domain: {session.domain}")
    
    # Store in both global and domain-specific databases
    state['sessions_db'][session.user_id].append(session)
    if session.domain:
        state['domain_sessions_db'][session.user_id][session.domain].append(session)
    
    print(f"✓ Session logged")
    print(f"{'─'*70}\n")

In [53]:
# ==================== MAIN ENTRY POINT ====================

def recommend_questions(state: Dict, user: UserProfile, k: int = 10, chosen_domain: str = None) -> Dict:
    """Main entry point - Enhanced Cold Start with Profile Building"""
    # Store user profile
    state['user_profiles'][user._id] = user
    
    # Check if first session for this domain
    if chosen_domain:
        sessions = state['domain_sessions_db'][user._id].get(chosen_domain, [])
    else:
        sessions = state['sessions_db'].get(user._id, [])
    
    is_first_session = len(sessions) == 0
    
    return enhanced_cold_start_flow(state, user, k, is_first_session, chosen_domain)


In [54]:
# ==================== EXAMPLE USAGE ====================

# Create sample questions for multiple domains
questions = [
    # Java questions
    Question(
        _id="q_java_0003", text="Java threading concepts",
        tags=["java", "concurrency"], domain="java",
        difficulty=4, estimated_time_sec=150
    ),
    Question(
        _id="q_java_0004", text="Exception handling in Java",
        tags=["java", "error-handling"], domain="java",
        difficulty=2, estimated_time_sec=80
    ),
    # Python questions
    Question(
        _id="q_py_0001", text="Python decorators explained",
        tags=["python", "advanced"], domain="python",
        difficulty=3, estimated_time_sec=100
    ),
    Question(
        _id="q_py_0002", text="List comprehensions in Python",
        tags=["python", "data-structures"], domain="python",
        difficulty=2, estimated_time_sec=70
    ),
    Question(
        _id="q_py_0003", text="Python generators and iterators",
        tags=["python", "advanced"], domain="python",
        difficulty=4, estimated_time_sec=130
    ),
    Question(
        _id="q_py_0004", text="Python async/await",
        tags=["python", "concurrency"], domain="python",
        difficulty=4, estimated_time_sec=140
    ),
    # ML questions
    Question(
        _id="q_ml_0001", text="Explain TF-IDF",
        tags=["nlp", "information-retrieval"], domain="ml",
        difficulty=2, estimated_time_sec=90
    ),
    Question(
        _id="q_ml_0002", text="Binary search algorithm",
        tags=["data-structures", "algorithms"], domain="ml",
        difficulty=1, estimated_time_sec=60
    ),
    Question(
        _id="q_ml_0003", text="Backpropagation explained",
        tags=["deep-learning", "tensorflow"], domain="ml",
        difficulty=3, estimated_time_sec=120
    ),
    Question(
        _id="q_ml_0004", text="Word embedding concepts",
        tags=["nlp", "deep-learning"], domain="ml",
        difficulty=2, estimated_time_sec=100
    ),
    Question(
        _id="q_ml_0005", text="Neural network in Python",
        tags=["python", "deep-learning"], domain="ml",
        difficulty=3, estimated_time_sec=180
    ),
    Question(
        _id="q_ml_0006", text="Data preprocessing techniques",
        tags=["python", "data-structures"], domain="ml",
        difficulty=2, estimated_time_sec=90
    ),
]

In [55]:
 # Create user
user = UserProfile(
    _id="user_123", name="Pavan", email="pavan@example.com",
    skills=["python", "java", "nlp", "tensorflow", "data-structures", "oop"],
    skill_levels={}, years_experience=2.5, education="B.Tech",
    target_domains=["java", "python", "ml"], target_companies=["CompanyA"],
    created_at="2025-09-01T12:00:00Z",
    last_active="2025-10-04T10:00:00Z"
)

In [57]:
# Initialize system
state = initialize_system(questions)
    
print("=" * 70)
print("DOMAIN-SPECIFIC LEARNING SYSTEM - DEMONSTRATION")
print("=" * 70)


[INIT] Enhanced Interview System Initialized
[INIT] Total Questions in Bank: 14

DOMAIN-SPECIFIC LEARNING SYSTEM - DEMONSTRATION


In [58]:
# ===== JAVA - FIRST SESSION =====
print("\n[DEMO] JAVA DOMAIN - FIRST SESSION")
print("=" * 70)
java_recommendations = recommend_questions(state, user, k=3, chosen_domain="java")
print(json.dumps(java_recommendations, indent=2))
    
# Simulate Java session responses
print("\n[DEMO] Simulating Java session responses...")
    
java_session = Session(
    _id="session_java_001", user_id="user_123",
    context={"company": "CompanyA", "domain": "java"},
    questions=[],
    created_at="2025-10-04T10:00:00Z",
    metrics_snapshot={},
    total_time_sec=300,
    questions_skipped=1,
    domain="java"
)
    
java_responses = [
    QuestionResponse(
        _id="resp_java_001", session_id="session_java_001", user_id="user_123",
        question_id="q_java_0001", response_text="Java inheritance explanation...",
        response_audio_features={"wpm": 105, "silence_ratio": 0.18},
        content_metrics={"relevance": 0.5, "grammar": 0.6},
        final_score=0.40,
        timestamp="2025-10-04T10:02:00Z",
        time_taken_sec=110,
        skipped=False,
        confidence_rating=2,
        keyword_matches=["inheritance", "extends"],
        domain="java"
    ),
    QuestionResponse(
        _id="resp_java_002", session_id="session_java_001", user_id="user_123",
        question_id="q_java_0002", response_text="",
        response_audio_features={},
        content_metrics={},
        final_score=0.0,
        timestamp="2025-10-04T10:04:00Z",
        time_taken_sec=5,
        skipped=True,
        confidence_rating=None,
        keyword_matches=[],
        domain="java"
    ),
    QuestionResponse(
        _id="resp_java_003", session_id="session_java_001", user_id="user_123",
        question_id="q_java_0004", response_text="Exception handling explanation...",
        response_audio_features={"wpm": 115, "silence_ratio": 0.12},
        content_metrics={"relevance": 0.7, "grammar": 0.75},
        final_score=0.65,
        timestamp="2025-10-04T10:06:00Z",
        time_taken_sec=85,
        skipped=False,
        confidence_rating=3,
        keyword_matches=["try", "catch", "finally", "throws"],
        domain="java"
    ),
]
    
log_session(state, java_session)
for response in java_responses:
    log_response(state, response)
    
build_user_profile_from_session(state, "user_123", "session_java_001", "java")


[DEMO] JAVA DOMAIN - FIRST SESSION

[ENHANCED COLD START] Profile-Building Flow
[0.1] User: user_123
[0.2] Is First Session: True
[0.3] Chosen Domain: java

[PHASE 1] First Session - Comprehensive Assessment
──────────────────────────────────────────────────────────────────────
[1.1] Goal: Assess all user skills comprehensively
[1.2] User Skills to Assess: ['python', 'java', 'nlp', 'tensorflow', 'data-structures', 'oop']

[1.3] Filtering by domain: ['java']
      ✓ 4 questions in target domains

[1.4] Creating diverse question set...
      - Skill Coverage Analysis:
        java: 4 questions available
        oop: 1 questions available
        data-structures: 1 questions available
      - Target: 1 questions per skill
      ✓ Selected 3 diverse questions

[1.5] Scoring questions for initial assessment...

      Question 1/3: q_java_0001
      - Tags: ['java', 'oop']
      - Difficulty: 2
      - Skill Overlap: 1.00
      - Difficulty Factor: 1.00
      - Final Score: 1.00

      Ques

In [59]:
# ===== PYTHON - FIRST SESSION (Independent from Java) =====
print("\n[DEMO] PYTHON DOMAIN - FIRST SESSION (Independent)")
print("=" * 70)
python_recommendations = recommend_questions(state, user, k=3, chosen_domain="python")
print(json.dumps(python_recommendations, indent=2))
    
# Simulate Python session responses
print("\n[DEMO] Simulating Python session responses...")
    
python_session = Session(
    _id="session_py_001", user_id="user_123",
    context={"company": "CompanyA", "domain": "python"},
    questions=[],
    created_at="2025-10-04T11:00:00Z",
    metrics_snapshot={},
    total_time_sec=280,
    questions_skipped=0,
    domain="python"
)
    
python_responses = [
    QuestionResponse(
        _id="resp_py_001", session_id="session_py_001", user_id="user_123",
        question_id="q_py_0001", response_text="Decorators explanation...",
        response_audio_features={"wpm": 120, "silence_ratio": 0.10},
        content_metrics={"relevance": 0.85, "grammar": 0.9},
        final_score=0.80,
        timestamp="2025-10-04T11:02:00Z",
        time_taken_sec=95,
        skipped=False,
        confidence_rating=4,
        keyword_matches=["decorator", "wrapper", "@", "function"],
        domain="python"
    ),
    QuestionResponse(
        _id="resp_py_002", session_id="session_py_001", user_id="user_123",
        question_id="q_py_0002", response_text="List comprehension explanation...",
        response_audio_features={"wpm": 125, "silence_ratio": 0.08},
        content_metrics={"relevance": 0.9, "grammar": 0.95},
        final_score=0.88,
        timestamp="2025-10-04T11:04:00Z",
        time_taken_sec=65,
        skipped=False,
        confidence_rating=5,
        keyword_matches=["comprehension", "list", "for", "if", "syntax"],
        domain="python"
    ),
    QuestionResponse(
        _id="resp_py_003", session_id="session_py_001", user_id="user_123",
        question_id="q_py_0003", response_text="Generators explanation...",
        response_audio_features={"wpm": 118, "silence_ratio": 0.12},
        content_metrics={"relevance": 0.75, "grammar": 0.8},
        final_score=0.72,
        timestamp="2025-10-04T11:07:00Z",
        time_taken_sec=120,
        skipped=False,
        confidence_rating=4,
        keyword_matches=["yield", "generator", "iterator", "lazy"],
        domain="python"
    ),
]
    
log_session(state, python_session)
for response in python_responses:
    log_response(state, response)
    
build_user_profile_from_session(state, "user_123", "session_py_001", "python")


[DEMO] PYTHON DOMAIN - FIRST SESSION (Independent)

[ENHANCED COLD START] Profile-Building Flow
[0.1] User: user_123
[0.2] Is First Session: True
[0.3] Chosen Domain: python

[PHASE 1] First Session - Comprehensive Assessment
──────────────────────────────────────────────────────────────────────
[1.1] Goal: Assess all user skills comprehensively
[1.2] User Skills to Assess: ['python', 'java', 'nlp', 'tensorflow', 'data-structures', 'oop']

[1.3] Filtering by domain: ['python']
      ✓ 4 questions in target domains

[1.4] Creating diverse question set...
      - Skill Coverage Analysis:
        python: 4 questions available
        data-structures: 1 questions available
      - Target: 1 questions per skill
      ✓ Selected 3 diverse questions

[1.5] Scoring questions for initial assessment...

      Question 1/3: q_py_0001
      - Tags: ['python', 'advanced']
      - Difficulty: 3
      - Skill Overlap: 0.50
      - Difficulty Factor: 1.00
      - Final Score: 0.50

      Question 2/3

In [60]:
# ===== JAVA - SECOND SESSION (Uses only Java history) =====
print("\n[DEMO] JAVA DOMAIN - SECOND SESSION (Profile-Based)")
print("=" * 70)
java_recommendations_2 = recommend_questions(state, user, k=3, chosen_domain="java")
print(json.dumps(java_recommendations_2, indent=2))


[DEMO] JAVA DOMAIN - SECOND SESSION (Profile-Based)

[ENHANCED COLD START] Profile-Building Flow
[0.1] User: user_123
[0.2] Is First Session: False
[0.3] Chosen Domain: java

[PHASE 2] Profile-Based Recommendations
──────────────────────────────────────────────────────────────────────
[2.1] Analyzing User Profile for Domain: java...

[2.2] Skill Analysis Summary:
      - data-structures: WEAKER
        Avg Score: 0.00, Questions: 1, Avg Time: 5s
      - java: WEAKER
        Avg Score: 0.35, Questions: 3, Avg Time: 67s
      - oop: WEAKER
        Avg Score: 0.40, Questions: 1, Avg Time: 110s
      - error-handling: MODERATE
        Avg Score: 0.65, Questions: 1, Avg Time: 85s

[2.3] Skill Categorization:
      - Weaker Skills (need improvement): ['data-structures', 'java', 'oop']
      - Stronger Skills (validation): []

[2.4] Difficulty Adaptation:
      - Last Score: 0.65
      - Adjustment: +0
      - Target Difficulty: 2

[2.5] Filtering Candidate Questions...
      - Previously At

In [61]:
# ===== PYTHON - SECOND SESSION (Uses only Python history) =====
print("\n[DEMO] PYTHON DOMAIN - SECOND SESSION (Profile-Based)")
print("=" * 70)
python_recommendations_2 = recommend_questions(state, user, k=3, chosen_domain="python")
print(json.dumps(python_recommendations_2, indent=2))


[DEMO] PYTHON DOMAIN - SECOND SESSION (Profile-Based)

[ENHANCED COLD START] Profile-Building Flow
[0.1] User: user_123
[0.2] Is First Session: False
[0.3] Chosen Domain: python

[PHASE 2] Profile-Based Recommendations
──────────────────────────────────────────────────────────────────────
[2.1] Analyzing User Profile for Domain: python...

[2.2] Skill Analysis Summary:
      - advanced: STRONGER
        Avg Score: 0.76, Questions: 2, Avg Time: 108s
      - python: STRONGER
        Avg Score: 0.80, Questions: 3, Avg Time: 93s
      - data-structures: STRONGER
        Avg Score: 0.88, Questions: 1, Avg Time: 65s

[2.3] Skill Categorization:
      - Weaker Skills (need improvement): []
      - Stronger Skills (validation): ['advanced', 'python', 'data-structures']

[2.4] Difficulty Adaptation:
      - Last Score: 0.72
      - Adjustment: +1
      - Target Difficulty: 3

[2.5] Filtering Candidate Questions...
      - Previously Attempted: 3
      - Weaker Skills Candidates: 0
      - Stro

In [62]:
# ===== ML DOMAIN - FIRST SESSION (Independent) =====
print("\n[DEMO] ML DOMAIN - FIRST SESSION (New Domain)")
print("=" * 70)
ml_recommendations = recommend_questions(state, user, k=3, chosen_domain="ml")
print(json.dumps(ml_recommendations, indent=2))


[DEMO] ML DOMAIN - FIRST SESSION (New Domain)

[ENHANCED COLD START] Profile-Building Flow
[0.1] User: user_123
[0.2] Is First Session: True
[0.3] Chosen Domain: ml

[PHASE 1] First Session - Comprehensive Assessment
──────────────────────────────────────────────────────────────────────
[1.1] Goal: Assess all user skills comprehensively
[1.2] User Skills to Assess: ['python', 'java', 'nlp', 'tensorflow', 'data-structures', 'oop']

[1.3] Filtering by domain: ['ml']
      ✓ 6 questions in target domains

[1.4] Creating diverse question set...
      - Skill Coverage Analysis:
        nlp: 2 questions available
        data-structures: 2 questions available
        tensorflow: 1 questions available
        python: 2 questions available
      - Target: 1 questions per skill
      ✓ Selected 3 diverse questions

[1.5] Scoring questions for initial assessment...

      Question 1/3: q_ml_0005
      - Tags: ['python', 'deep-learning']
      - Difficulty: 3
      - Skill Overlap: 0.50
      - 

In [63]:
# Show domain-specific profiles
print("\n[DEMO] DOMAIN-SPECIFIC USER PROFILES")
print("=" * 70)
user_profile = state['user_profiles'].get("user_123")
if user_profile:
    print("\n--- JAVA PROFILE ---")
    if "java" in user_profile.domain_specific_skill_metadata:
        print(f"Weaker Skills: {user_profile.domain_specific_weaker_skills.get('java', [])}")
        print(f"Stronger Skills: {user_profile.domain_specific_stronger_skills.get('java', [])}")
        print("\nDetailed Metrics:")
        for skill, metadata in user_profile.domain_specific_skill_metadata["java"].items():
            print(f"  {skill}:")
            print(f"    Category: {metadata['category']}")
            print(f"    Avg Score: {metadata['avg_score']:.1%}")
            print(f"    Questions: {metadata['total_questions']}")
            print(f"    Avg Time: {metadata['avg_time_sec']:.0f}s")
        
    print("\n--- PYTHON PROFILE ---")
    if "python" in user_profile.domain_specific_skill_metadata:
        print(f"Weaker Skills: {user_profile.domain_specific_weaker_skills.get('python', [])}")
        print(f"Stronger Skills: {user_profile.domain_specific_stronger_skills.get('python', [])}")
        print("\nDetailed Metrics:")
        for skill, metadata in user_profile.domain_specific_skill_metadata["python"].items():
            print(f"  {skill}:")
            print(f"    Category: {metadata['category']}")
            print(f"    Avg Score: {metadata['avg_score']:.1%}")
            print(f"    Questions: {metadata['total_questions']}")
            print(f"    Avg Time: {metadata['avg_time_sec']:.0f}s")


[DEMO] DOMAIN-SPECIFIC USER PROFILES

--- JAVA PROFILE ---
Weaker Skills: ['data-structures', 'java', 'oop']
Stronger Skills: []

Detailed Metrics:
  data-structures:
    Category: weaker
    Avg Score: 0.0%
    Questions: 1
    Avg Time: 5s
  java:
    Category: weaker
    Avg Score: 35.0%
    Questions: 3
    Avg Time: 67s
  oop:
    Category: weaker
    Avg Score: 40.0%
    Questions: 1
    Avg Time: 110s
  error-handling:
    Category: moderate
    Avg Score: 65.0%
    Questions: 1
    Avg Time: 85s

--- PYTHON PROFILE ---
Weaker Skills: []
Stronger Skills: ['advanced', 'python', 'data-structures']

Detailed Metrics:
  advanced:
    Category: stronger
    Avg Score: 76.0%
    Questions: 2
    Avg Time: 108s
  python:
    Category: stronger
    Avg Score: 80.0%
    Questions: 3
    Avg Time: 93s
  data-structures:
    Category: stronger
    Avg Score: 88.0%
    Questions: 1
    Avg Time: 65s


In [68]:
import json
from dataclasses import asdict

profile_dict = asdict(user)   # Convert dataclass to dictionary
pretty_json = json.dumps(profile_dict, indent=4)  # Convert dict to JSON with indentation

print(pretty_json)

{
    "_id": "user_123",
    "name": "Pavan",
    "email": "pavan@example.com",
    "skills": [
        "python",
        "java",
        "nlp",
        "tensorflow",
        "data-structures",
        "oop"
    ],
    "skill_levels": {},
    "years_experience": 2.5,
    "education": "B.Tech",
    "target_domains": [
        "java",
        "python",
        "ml"
    ],
    "target_companies": [
        "CompanyA"
    ],
    "created_at": "2025-09-01T12:00:00Z",
    "last_active": "2025-10-04T10:00:00Z",
    "weaker_skills": [],
    "stronger_skills": [],
    "skill_metadata": {},
    "domain_specific_weaker_skills": {
        "java": [
            "data-structures",
            "java",
            "oop"
        ],
        "python": []
    },
    "domain_specific_stronger_skills": {
        "java": [],
        "python": [
            "advanced",
            "python",
            "data-structures"
        ]
    },
    "domain_specific_skill_metadata": {
        "java": {
            "da

In [1]:
"""
Enhanced Personalized Interview System - Multi-Modal Assessment

Incorporates audio analysis, content metrics, and question type performance
"""

from typing import List, Dict, Set, Optional, Tuple
from datetime import datetime
from dataclasses import dataclass, field
from collections import defaultdict
import json

# ==================== DATA STRUCTURES ====================

@dataclass
class AudioMetrics:
    """Detailed audio analysis from speech"""
    silence_ratio: float
    speech_rate_wpm: float
    pitch_variation: float
    clarity_score: float
    pacing_consistency: float
    expressiveness: float
    confidence_score: float
    transcription_confidence: str

@dataclass
class ContentMetrics:
    """Content quality assessment"""
    grammar_score: float
    relevance_score: float
    completeness_score: float
    overall_score: float
    missing_points: List[str]
    grammar_issues: List[str]

@dataclass
class UserProfile:
    """Enhanced User Profile with Multi-Modal Performance"""
    _id: str
    name: str
    email: str
    skills: List[str]
    skill_levels: Dict[str, int]
    years_experience: float
    education: str
    target_domains: List[str]
    target_companies: List[str]
    created_at: str
    last_active: str
    
    # Performance by question type
    question_type_performance: Dict[str, Dict] = field(default_factory=dict)
    
    # Skill-specific performance
    weaker_skills: List[str] = field(default_factory=list)
    stronger_skills: List[str] = field(default_factory=list)
    skill_metadata: Dict[str, Dict] = field(default_factory=dict)
    
    # Domain-specific tracking
    domain_specific_weaker_skills: Dict[str, List[str]] = field(default_factory=dict)
    domain_specific_stronger_skills: Dict[str, List[str]] = field(default_factory=dict)
    domain_specific_skill_metadata: Dict[str, Dict[str, Dict]] = field(default_factory=dict)
    
    # Communication metrics (for voice-based questions)
    avg_communication_score: float = 0.0
    communication_strengths: List[str] = field(default_factory=list)
    communication_weaknesses: List[str] = field(default_factory=list)

@dataclass
class Question:
    """Enhanced Question with Type and Difficulty Metadata"""
    _id: str
    text: str
    tags: List[str]
    domain: str
    difficulty: int
    estimated_time_sec: int
    question_type: str  # 'mcq', 'voice', 'coding', 'subjective'
    
    embedding: List[float] = field(default_factory=list)
    created_by: str = "editor"
    popularity: int = 0
    company_specific: List[str] = field(default_factory=list)
    last_updated: str = ""
    
    # Type-specific requirements
    requires_audio_analysis: bool = False
    requires_code_execution: bool = False
    has_multiple_correct_answers: bool = False

@dataclass
class QuestionResponse:
    """Enhanced Response with Multi-Modal Assessment"""
    _id: str
    session_id: str
    user_id: str
    question_id: str
    question_type: str
    response_text: str
    
    # Audio metrics (for voice-based)
    audio_metrics: Optional[AudioMetrics] = None
    
    # Content evaluation
    content_metrics: Optional[ContentMetrics] = None
    
    # Combined scores
    final_score: float = 0.0
    
    # Metadata
    timestamp: str = ""
    time_taken_sec: int = 0
    skipped: bool = False
    confidence_rating: Optional[int] = None
    keyword_matches: List[str] = field(default_factory=list)
    domain: str = ""
    
    # Performance breakdown
    technical_score: float = 0.0
    communication_score: float = 0.0
    completeness_score: float = 0.0

@dataclass
class Session:
    """Enhanced Session with Question Type Tracking"""
    _id: str
    user_id: str
    context: Dict[str, str]
    questions: List[Dict[str, str]]
    created_at: str
    metrics_snapshot: Dict[str, float]
    
    skill_performance: Dict[str, List[float]] = field(default_factory=dict)
    question_type_performance: Dict[str, List[float]] = field(default_factory=dict)
    
    weaker_skills_identified: List[str] = field(default_factory=list)
    stronger_skills_identified: List[str] = field(default_factory=list)
    
    total_time_sec: int = 0
    questions_skipped: int = 0
    domain: str = ""

@dataclass
class SkillAnalysis:
    """Enhanced skill analysis with multi-modal metrics"""
    skill: str
    avg_score: float
    total_questions: int
    avg_time_sec: float
    skip_rate: float
    avg_confidence: Optional[float]
    keyword_match_rate: float
    category: str
    
    # Question type breakdown
    performance_by_type: Dict[str, float] = field(default_factory=dict)
    
    # Communication metrics (for voice questions)
    avg_communication_score: Optional[float] = None
    clarity_issues: List[str] = field(default_factory=list)
    content_issues: List[str] = field(default_factory=list)

@dataclass
class RecommendedQuestion:
    """Enhanced recommendation with reasoning"""
    q_id: str
    score: float
    reasoning: str = ""
    recommended_type: str = ""
    priority_factors: List[str] = field(default_factory=list)

# ==================== MULTI-MODAL ANALYSIS ====================

def analyze_audio_performance(audio_metrics: AudioMetrics) -> Dict[str, any]:
    """Analyze audio performance and identify strengths/weaknesses"""
    strengths = []
    weaknesses = []
    
    # Silence ratio analysis
    if audio_metrics.silence_ratio < 15:
        strengths.append("confident_delivery")
    elif audio_metrics.silence_ratio > 25:
        weaknesses.append("excessive_pauses")
    
    # Speech rate analysis
    if 140 <= audio_metrics.speech_rate_wpm <= 160:
        strengths.append("optimal_pace")
    elif audio_metrics.speech_rate_wpm < 100:
        weaknesses.append("slow_speech")
    elif audio_metrics.speech_rate_wpm > 180:
        weaknesses.append("rushed_speech")
    
    # Clarity analysis
    if audio_metrics.clarity_score >= 95:
        strengths.append("clear_articulation")
    elif audio_metrics.clarity_score < 85:
        weaknesses.append("unclear_speech")
    
    # Expressiveness analysis
    if audio_metrics.expressiveness >= 60:
        strengths.append("expressive_communication")
    elif audio_metrics.expressiveness < 40:
        weaknesses.append("monotone_delivery")
    
    # Overall confidence
    if audio_metrics.confidence_score >= 70:
        strengths.append("high_confidence")
    elif audio_metrics.confidence_score < 50:
        weaknesses.append("low_confidence")
    
    return {
        'overall_score': audio_metrics.confidence_score,
        'strengths': strengths,
        'weaknesses': weaknesses,
        'needs_improvement': len(weaknesses) > 2
    }

def analyze_content_quality(content_metrics: ContentMetrics) -> Dict[str, any]:
    """Analyze content quality and completeness"""
    strengths = []
    issues = []
    
    # Grammar analysis
    if content_metrics.grammar_score >= 95:
        strengths.append("excellent_grammar")
    elif content_metrics.grammar_score < 80:
        issues.extend(content_metrics.grammar_issues)
    
    # Relevance analysis
    if content_metrics.relevance_score >= 90:
        strengths.append("highly_relevant")
    elif content_metrics.relevance_score < 70:
        issues.append("off_topic_content")
    
    # Completeness analysis
    if content_metrics.completeness_score >= 80:
        strengths.append("comprehensive_answer")
    elif content_metrics.completeness_score < 60:
        issues.append(f"missed_points: {', '.join(content_metrics.missing_points[:3])}")
    
    return {
        'overall_score': content_metrics.overall_score,
        'strengths': strengths,
        'issues': issues,
        'needs_improvement': len(issues) > 1
    }

def calculate_multi_modal_score(response: QuestionResponse) -> Tuple[float, Dict]:
    """Calculate comprehensive score from multi-modal data"""
    scores = {}
    weights = {}
    
    if response.question_type == 'voice':
        # Voice questions: balance technical content and communication
        if response.content_metrics:
            scores['technical'] = response.content_metrics.overall_score / 100
            weights['technical'] = 0.6
        
        if response.audio_metrics:
            scores['communication'] = response.audio_metrics.confidence_score / 100
            weights['communication'] = 0.4
    
    elif response.question_type == 'mcq':
        # MCQ: focus on accuracy and speed
        scores['accuracy'] = response.final_score
        weights['accuracy'] = 0.8
        
        # Time factor (faster is better, but not rushed)
        if response.time_taken_sec > 0:
            time_factor = min(1.0, 60 / response.time_taken_sec)
            scores['efficiency'] = time_factor
            weights['efficiency'] = 0.2
    
    elif response.question_type == 'coding':
        # Coding: correctness, efficiency, and code quality
        if response.content_metrics:
            scores['correctness'] = response.content_metrics.overall_score / 100
            weights['correctness'] = 0.7
            scores['completeness'] = response.content_metrics.completeness_score / 100
            weights['completeness'] = 0.3
    
    elif response.question_type == 'subjective':
        # Subjective: content quality, completeness, clarity
        if response.content_metrics:
            scores['content'] = response.content_metrics.relevance_score / 100
            weights['content'] = 0.5
            scores['completeness'] = response.content_metrics.completeness_score / 100
            weights['completeness'] = 0.3
            scores['grammar'] = response.content_metrics.grammar_score / 100
            weights['grammar'] = 0.2
    
    # Calculate weighted average
    if scores and weights:
        total_weight = sum(weights.values())
        final_score = sum(scores[k] * weights[k] for k in scores) / total_weight
    else:
        final_score = response.final_score
    
    return final_score, scores

# ==================== ENHANCED SKILL ANALYSIS ====================

def analyze_user_skills_multimodal(state: Dict, user_id: str, domain: str = None) -> List[SkillAnalysis]:
    """Enhanced skill analysis with question type performance"""
    if domain:
        responses = state['domain_responses_db'][user_id].get(domain, [])
    else:
        responses = state['responses_db'].get(user_id, [])
    
    if not responses:
        return []
    
    skill_data = defaultdict(lambda: {
        'scores': [],
        'times': [],
        'skips': [],
        'confidences': [],
        'keyword_matches': [],
        'type_scores': defaultdict(list),
        'communication_scores': [],
        'content_issues': []
    })
    
    # Aggregate data per skill
    for response in responses:
        question = get_question_by_id(state['question_bank'], response.question_id)
        if not question:
            continue
        
        # Calculate multi-modal score
        final_score, score_breakdown = calculate_multi_modal_score(response)
        
        for tag in question.tags:
            skill_data[tag]['scores'].append(final_score)
            skill_data[tag]['times'].append(response.time_taken_sec)
            skill_data[tag]['skips'].append(1 if response.skipped else 0)
            skill_data[tag]['type_scores'][response.question_type].append(final_score)
            
            if response.confidence_rating:
                skill_data[tag]['confidences'].append(response.confidence_rating)
            
            if response.keyword_matches:
                skill_data[tag]['keyword_matches'].append(len(response.keyword_matches))
            
            # Communication analysis for voice questions
            if response.audio_metrics:
                audio_analysis = analyze_audio_performance(response.audio_metrics)
                skill_data[tag]['communication_scores'].append(audio_analysis['overall_score'])
                if audio_analysis['weaknesses']:
                    skill_data[tag]['content_issues'].extend(audio_analysis['weaknesses'])
            
            # Content analysis
            if response.content_metrics:
                content_analysis = analyze_content_quality(response.content_metrics)
                if content_analysis['issues']:
                    skill_data[tag]['content_issues'].extend(content_analysis['issues'])
    
    # Create enhanced analyses
    analyses = []
    for skill, data in skill_data.items():
        if not data['scores']:
            continue
        
        avg_score = sum(data['scores']) / len(data['scores'])
        avg_time = sum(data['times']) / len(data['times']) if data['times'] else 0
        skip_rate = sum(data['skips']) / len(data['skips'])
        
        # Calculate performance by question type
        perf_by_type = {}
        for q_type, scores in data['type_scores'].items():
            if scores:
                perf_by_type[q_type] = sum(scores) / len(scores)
        
        # Communication score (for voice questions)
        avg_comm_score = None
        if data['communication_scores']:
            avg_comm_score = sum(data['communication_scores']) / len(data['communication_scores'])
        
        # Categorize skill with more nuance
        if avg_score < 0.5 or skip_rate > 0.3:
            category = 'weaker'
        elif avg_score > 0.75 and skip_rate < 0.1:
            category = 'stronger'
        else:
            category = 'moderate'
        
        # Identify specific issues
        unique_issues = list(set(data['content_issues']))[:3]
        
        analyses.append(SkillAnalysis(
            skill=skill,
            avg_score=avg_score,
            total_questions=len(data['scores']),
            avg_time_sec=avg_time,
            skip_rate=skip_rate,
            avg_confidence=sum(data['confidences']) / len(data['confidences']) if data['confidences'] else None,
            keyword_match_rate=sum(data['keyword_matches']) / len(data['keyword_matches']) if data['keyword_matches'] else 0,
            category=category,
            performance_by_type=perf_by_type,
            avg_communication_score=avg_comm_score,
            content_issues=unique_issues
        ))
    
    return sorted(analyses, key=lambda x: x.avg_score)

# ==================== INTELLIGENT QUESTION TYPE SELECTION ====================

def select_optimal_question_type(skill_analysis: SkillAnalysis, user_profile: UserProfile) -> str:
    """Select best question type based on user's performance patterns"""
    
    # If user struggles with voice questions, give more practice
    if 'voice' in skill_analysis.performance_by_type:
        voice_score = skill_analysis.performance_by_type['voice']
        if voice_score < 0.6 and 'unclear_speech' in skill_analysis.clarity_issues:
            return 'voice'  # Need communication practice
    
    # If user excels at MCQs but struggles with depth, give subjective
    if 'mcq' in skill_analysis.performance_by_type and 'subjective' in skill_analysis.performance_by_type:
        mcq_score = skill_analysis.performance_by_type['mcq']
        subj_score = skill_analysis.performance_by_type.get('subjective', 0)
        if mcq_score > 0.8 and subj_score < 0.6:
            return 'subjective'  # Test deeper understanding
    
    # For weaker skills, start with MCQs then progress
    if skill_analysis.category == 'weaker':
        if not skill_analysis.performance_by_type:
            return 'mcq'  # Start with basics
        elif 'mcq' in skill_analysis.performance_by_type and skill_analysis.performance_by_type['mcq'] > 0.7:
            return 'subjective'  # Ready for deeper questions
    
    # For stronger skills, challenge with voice or coding
    if skill_analysis.category == 'stronger':
        if 'coding' in skill_analysis.performance_by_type:
            return 'coding'  # Maintain technical depth
        return 'voice'  # Test explanation ability
    
    # Default: balance between types
    return 'subjective'

def score_questions_with_type_matching(questions: List[Question], 
                                       skill_analyses: List[SkillAnalysis],
                                       user_profile: UserProfile,
                                       focus_type: str) -> List[Tuple[Question, float, str]]:
    """Enhanced scoring with question type optimization"""
    scored = []
    skill_map = {a.skill: a for a in skill_analyses}
    
    for question in questions:
        question_skills = [s for s in question.tags if s in skill_map]
        if not question_skills:
            continue
        
        # Base score from skill performance
        avg_skill_score = sum(skill_map[s].avg_score for s in question_skills) / len(question_skills)
        
        # Get optimal question type for these skills
        primary_skill = question_skills[0]
        optimal_type = select_optimal_question_type(skill_map[primary_skill], user_profile)
        
        # Type matching bonus
        type_match_bonus = 0.3 if question.question_type == optimal_type else 0
        
        # Performance-based priority
        if focus_type == 'weaker':
            priority = (1 - avg_skill_score) + type_match_bonus
            
            # Add specific issue targeting
            issues = skill_map[primary_skill].content_issues
            issue_str = f" (addressing: {', '.join(issues[:2])})" if issues else ""
            
            reasoning = (
                f"{question.question_type.upper()}: Focus on {', '.join(question_skills)} "
                f"(current: {avg_skill_score:.0%}){issue_str}"
            )
        else:
            priority = avg_skill_score + type_match_bonus
            reasoning = (
                f"{question.question_type.upper()}: Validate {', '.join(question_skills)} "
                f"mastery (strong: {avg_skill_score:.0%})"
            )
        
        # Priority factors for transparency
        factors = [f"skill_match", f"type_{question.question_type}"]
        if type_match_bonus > 0:
            factors.append("optimal_format")
        
        scored.append((question, priority, reasoning, factors))
    
    return scored

# ==================== HELPER FUNCTIONS ====================

def initialize_system(question_bank: List[Question]) -> Dict:
    """Initialize system state"""
    print(f"\n{'='*70}")
    print(f"[INIT] Multi-Modal Interview System Initialized")
    print(f"[INIT] Total Questions: {len(question_bank)}")
    
    # Count by type
    type_counts = defaultdict(int)
    for q in question_bank:
        type_counts[q.question_type] += 1
    
    print(f"[INIT] Question Types:")
    for q_type, count in type_counts.items():
        print(f"       - {q_type}: {count}")
    print(f"{'='*70}\n")
    
    return {
        'question_bank': question_bank,
        'sessions_db': defaultdict(list),
        'responses_db': defaultdict(list),
        'user_profiles': {},
        'domain_sessions_db': defaultdict(lambda: defaultdict(list)),
        'domain_responses_db': defaultdict(lambda: defaultdict(list))
    }

def get_question_by_id(question_bank: List[Question], question_id: str) -> Optional[Question]:
    """Retrieve question by ID"""
    for q in question_bank:
        if q._id == question_id:
            return q
    return None

def filter_by_domain_and_type(question_bank: List[Question], 
                               target_domains: List[str],
                               preferred_types: List[str] = None) -> List[Question]:
    """Filter questions by domain and optionally by type"""
    filtered = [q for q in question_bank if q.domain in target_domains]
    
    if preferred_types:
        filtered = [q for q in filtered if q.question_type in preferred_types]
    
    return filtered

def get_attempted_questions(state: Dict, user_id: str, domain: str = None) -> Set[str]:
    """Get attempted questions"""
    if domain:
        responses = state['domain_responses_db'][user_id].get(domain, [])
    else:
        responses = state['responses_db'].get(user_id, [])
    return {response.question_id for response in responses}

# ==================== ENHANCED RECOMMENDATIONS ====================

def enhanced_recommendations(state: Dict, user: UserProfile, k: int, 
                            chosen_domain: str = None) -> Dict:
    """Generate intelligent multi-modal recommendations"""
    print(f"\n[ENHANCED RECOMMENDATIONS] Multi-Modal Analysis")
    print(f"{'─'*70}")
    
    # Analyze user performance
    skill_analyses = analyze_user_skills_multimodal(state, user._id, chosen_domain)
    
    if not skill_analyses:
        print(f"[INFO] No history found - generating baseline assessment")
        return baseline_assessment(state, user, k, chosen_domain)
    
    print(f"\n[SKILL ANALYSIS] Performance Summary:")
    for analysis in skill_analyses:
        print(f"\n  {analysis.skill} ({analysis.category.upper()}):")
        print(f"    Overall Score: {analysis.avg_score:.1%}")
        print(f"    Questions: {analysis.total_questions}")
        
        if analysis.performance_by_type:
            print(f"    By Type:")
            for q_type, score in analysis.performance_by_type.items():
                print(f"      - {q_type}: {score:.1%}")
        
        if analysis.content_issues:
            print(f"    Issues: {', '.join(analysis.content_issues)}")
    
    # Categorize skills
    weaker_skills = [a for a in skill_analyses if a.category == 'weaker']
    stronger_skills = [a for a in skill_analyses if a.category == 'stronger']
    
    print(f"\n[FOCUS AREAS]")
    print(f"  Weaker Skills: {[s.skill for s in weaker_skills]}")
    print(f"  Stronger Skills: {[s.skill for s in stronger_skills]}")
    
    # Get candidate questions
    target_domains = [chosen_domain] if chosen_domain else user.target_domains
    attempted = get_attempted_questions(state, user._id, chosen_domain)
    
    candidates = filter_by_domain_and_type(state['question_bank'], target_domains)
    candidates = [q for q in candidates if q._id not in attempted]
    
    # Score with type matching
    weaker_scored = score_questions_with_type_matching(
        candidates, weaker_skills, user, 'weaker'
    )
    stronger_scored = score_questions_with_type_matching(
        candidates, stronger_skills, user, 'stronger'
    )
    
    # Allocate 70% weaker, 30% stronger
    k_weaker = int(k * 0.7)
    k_stronger = k - k_weaker
    
    weaker_ranked = sorted(weaker_scored, key=lambda x: x[1], reverse=True)[:k_weaker]
    stronger_ranked = sorted(stronger_scored, key=lambda x: x[1], reverse=True)[:k_stronger]
    
    # Generate recommendations
    recommendations = []
    
    for q, score, reasoning, factors in weaker_ranked:
        recommendations.append(RecommendedQuestion(
            q_id=q._id,
            score=round(score, 2),
            reasoning=reasoning,
            recommended_type=q.question_type,
            priority_factors=factors
        ))
    
    for q, score, reasoning, factors in stronger_ranked:
        recommendations.append(RecommendedQuestion(
            q_id=q._id,
            score=round(score, 2),
            reasoning=reasoning,
            recommended_type=q.question_type,
            priority_factors=factors
        ))
    
    print(f"\n[RECOMMENDATIONS] {len(recommendations)} questions selected")
    for i, rec in enumerate(recommendations):
        print(f"\n  {i+1}. {rec.q_id} (Score: {rec.score})")
        print(f"     Type: {rec.recommended_type}")
        print(f"     Reason: {rec.reasoning}")
        print(f"     Factors: {', '.join(rec.priority_factors)}")
    
    output = {
        "user_id": user._id,
        "session_type": "multi_modal_adaptive",
        "domain": chosen_domain,
        "skill_analyses": [
            {
                "skill": a.skill,
                "category": a.category,
                "avg_score": round(a.avg_score, 2),
                "performance_by_type": {k: round(v, 2) for k, v in a.performance_by_type.items()},
                "issues": a.content_issues
            }
            for a in skill_analyses
        ],
        "questions_recommended": [
            {
                "q_id": r.q_id,
                "score": r.score,
                "type": r.recommended_type,
                "reasoning": r.reasoning,
                "priority_factors": r.priority_factors
            }
            for r in recommendations
        ]
    }
    
    print(f"{'='*70}\n")
    return output

def baseline_assessment(state: Dict, user: UserProfile, k: int, chosen_domain: str) -> Dict:
    """Initial assessment with diverse question types"""
    print(f"[BASELINE] Creating diverse assessment across question types")
    
    target_domains = [chosen_domain] if chosen_domain else user.target_domains
    candidates = filter_by_domain_and_type(state['question_bank'], target_domains)
    
    # Balance across types
    type_distribution = {'mcq': 0.3, 'subjective': 0.3, 'voice': 0.2, 'coding': 0.2}
    recommendations = []
    
    for q_type, ratio in type_distribution.items():
        type_questions = [q for q in candidates if q.question_type == q_type]
        count = int(k * ratio)
        
        for q in type_questions[:count]:
            recommendations.append(RecommendedQuestion(
                q_id=q._id,
                score=1.0,
                reasoning=f"Baseline {q_type} assessment for {', '.join(q.tags)}",
                recommended_type=q_type,
                priority_factors=["baseline", "skill_coverage"]
            ))
    
    return {
        "user_id": user._id,
        "session_type": "baseline_assessment",
        "domain": chosen_domain,
        "message": "Baseline assessment across all question types",
        "questions_recommended": [
            {
                "q_id": r.q_id,
                "score": r.score,
                "type": r.recommended_type,
                "reasoning": r.reasoning
            }
            for r in recommendations
        ]
    }

# ==================== MAIN ENTRY POINT ====================

def recommend_questions(state: Dict, user: UserProfile, k: int = 10, 
                       chosen_domain: str = None) -> Dict:
    """Main recommendation engine"""
    state['user_profiles'][user._id] = user
    return enhanced_recommendations(state, user, k, chosen_domain)

def log_response(state: Dict, response: QuestionResponse):
    """Log response with multi-modal data"""
    print(f"\n[RESPONSE] Logged: {response.question_id}")
    print(f"  Type: {response.question_type}")
    print(f"  Score: {response.final_score:.2f}")
    
    if response.audio_metrics:
        print(f"  Audio Score: {response.audio_metrics.confidence_score:.1f}")
    
    if response.content_metrics:
        print(f"  Content Score: {response.content_metrics.overall_score:.1f}")
    
    state['responses_db'][response.user_id].append(response)
    if response.domain:
        state['domain_responses_db'][response.user_id][response.domain].append(response)

# ==================== EXAMPLE USAGE ====================

if __name__ == "__main__":
    # Create sample questions with types
    questions = [
        Question(_id="q1", text="Java inheritance", tags=["java", "oop"], 
                domain="java", difficulty=2, estimated_time_sec=120, question_type="mcq"),
        Question(_id="q2", text="Explain polymorphism", tags=["java", "oop"], 
                domain="java", difficulty=3, estimated_time_sec=180, question_type="voice"),
        Question(_id="q3", text="Implement binary search", tags=["java", "algorithms"], 
                domain="java", difficulty=3, estimated_time_sec=300, question_type="coding"),
        Question(_id="q4", text="Python decorators", tags=["python", "advanced"], 
                domain="python", difficulty=3, estimated_time_sec=150, question_type="subjective"),
        Question(_id="q5", text="List comprehension", tags=["python", "basics"], 
                domain="python", difficulty=2, estimated_time_sec=90, question_type="mcq"),
    ]
    
    user = UserProfile(
        _id="user_001", name="Test User", email="test@example.com",
        skills=["java", "python", "oop", "algorithms"],
        skill_levels={}, years_experience=2.0, education="B.Tech",
        target_domains=["java", "python"], target_companies=["TechCorp"],
        created_at="2025-01-01T00:00:00Z", last_active="2025-01-15T00:00:00Z"
    )
    
    state = initialize_system(questions)
    
    print("\n" + "="*70)
    print("MULTI-MODAL QUESTION RECOMMENDATION DEMO")
    print("="*70)
    
    # First session - baseline
    print("\n[DEMO 1] First Session - Baseline Assessment")
    result1 = recommend_questions(state, user, k=5, chosen_domain="java")
    print(json.dumps(result1, indent=2))
    
    # Simulate responses with audio and content metrics
    print("\n[DEMO 2] Simulating Multi-Modal Responses")
    
    # Voice question response - struggling with communication
    audio_metrics_weak = AudioMetrics(
        silence_ratio=28.5,
        speech_rate_wpm=95.3,
        pitch_variation=25.4,
        clarity_score=82.1,
        pacing_consistency=15.3,
        expressiveness=38.2,
        confidence_score=42.8,
        transcription_confidence="medium"
    )
    
    content_metrics_weak = ContentMetrics(
        grammar_score=85.0,
        relevance_score=72.3,
        completeness_score=65.0,
        overall_score=74.1,
        missing_points=["key concept", "example"],
        grammar_issues=["Minor issues"]
    )
    
    response1 = QuestionResponse(
        _id="resp_001",
        session_id="session_001",
        user_id="user_001",
        question_id="q2",
        question_type="voice",
        response_text="Polymorphism allows objects...",
        audio_metrics=audio_metrics_weak,
        content_metrics=content_metrics_weak,
        final_score=0.68,
        timestamp="2025-01-15T10:00:00Z",
        time_taken_sec=195,
        domain="java"
    )
    log_response(state, response1)
    
    # MCQ response - strong performance
    response2 = QuestionResponse(
        _id="resp_002",
        session_id="session_001",
        user_id="user_001",
        question_id="q1",
        question_type="mcq",
        response_text="Option B",
        final_score=0.95,
        timestamp="2025-01-15T10:05:00Z",
        time_taken_sec=45,
        domain="java"
    )
    log_response(state, response2)
    
    # Coding response - moderate with content issues
    content_metrics_coding = ContentMetrics(
        grammar_score=100.0,
        relevance_score=88.5,
        completeness_score=70.0,
        overall_score=86.2,
        missing_points=["edge cases", "optimization"],
        grammar_issues=[]
    )
    
    response3 = QuestionResponse(
        _id="resp_003",
        session_id="session_001",
        user_id="user_001",
        question_id="q3",
        question_type="coding",
        response_text="def binary_search(arr, target)...",
        content_metrics=content_metrics_coding,
        final_score=0.78,
        timestamp="2025-01-15T10:15:00Z",
        time_taken_sec=280,
        domain="java"
    )
    log_response(state, response3)
    
    # Second session - adaptive recommendations
    print("\n[DEMO 3] Second Session - Adaptive Multi-Modal Recommendations")
    result2 = recommend_questions(state, user, k=5, chosen_domain="java")
    print(json.dumps(result2, indent=2))
    
    # Add excellent voice response to show improvement
    print("\n[DEMO 4] Simulating Improved Performance")
    
    audio_metrics_strong = AudioMetrics(
        silence_ratio=17.0,
        speech_rate_wpm=150.1,
        pitch_variation=39.6,
        clarity_score=98.7,
        pacing_consistency=78.5,
        expressiveness=59.2,
        confidence_score=82.9,
        transcription_confidence="high"
    )
    
    content_metrics_strong = ContentMetrics(
        grammar_score=100.0,
        relevance_score=95.7,
        completeness_score=92.0,
        overall_score=95.9,
        missing_points=[],
        grammar_issues=[]
    )
    
    response4 = QuestionResponse(
        _id="resp_004",
        session_id="session_002",
        user_id="user_001",
        question_id="q2",  # Same voice question, improved
        question_type="voice",
        response_text="Polymorphism is a fundamental OOP concept...",
        audio_metrics=audio_metrics_strong,
        content_metrics=content_metrics_strong,
        final_score=0.94,
        timestamp="2025-01-15T11:00:00Z",
        time_taken_sec=165,
        domain="java"
    )
    log_response(state, response4)
    
    # Third session - see adaptation to improvement
    print("\n[DEMO 5] Third Session - Recognizing Improvement")
    result3 = recommend_questions(state, user, k=5, chosen_domain="java")
    print(json.dumps(result3, indent=2))
    
    # Show detailed skill analysis
    print("\n[DEMO 6] Detailed Multi-Modal Skill Analysis")
    print("="*70)
    skill_analyses = analyze_user_skills_multimodal(state, "user_001", "java")
    
    for analysis in skill_analyses:
        print(f"\n{'─'*70}")
        print(f"Skill: {analysis.skill} ({analysis.category.upper()})")
        print(f"{'─'*70}")
        print(f"Overall Performance:")
        print(f"  Average Score: {analysis.avg_score:.1%}")
        print(f"  Total Questions: {analysis.total_questions}")
        print(f"  Skip Rate: {analysis.skip_rate:.1%}")
        print(f"  Avg Time: {analysis.avg_time_sec:.0f}s")
        
        if analysis.performance_by_type:
            print(f"\nPerformance by Question Type:")
            for q_type, score in sorted(analysis.performance_by_type.items()):
                print(f"  {q_type.upper():12s}: {score:.1%}")
        
        if analysis.avg_communication_score:
            print(f"\nCommunication Score: {analysis.avg_communication_score:.1f}/100")
        
        if analysis.content_issues:
            print(f"\nAreas for Improvement:")
            for issue in analysis.content_issues:
                print(f"  • {issue.replace('_', ' ').title()}")
        
        print(f"\nRecommendation:")
        optimal_type = select_optimal_question_type(analysis, user)
        print(f"  Next question type: {optimal_type.upper()}")
        if analysis.category == 'weaker':
            print(f"  Focus: Build foundational understanding")
        elif analysis.category == 'stronger':
            print(f"  Focus: Challenge with advanced problems")
        else:
            print(f"  Focus: Maintain steady progress")
    
    print("\n" + "="*70)
    print("DEMONSTRATION COMPLETE")
    print("="*70)
    print("\nKey Features Demonstrated:")
    print("✓ Multi-modal assessment (voice, content, timing)")
    print("✓ Question type optimization based on performance")
    print("✓ Adaptive difficulty and format selection")
    print("✓ Detailed issue tracking and targeting")
    print("✓ Communication skill analysis for voice questions")
    print("✓ Progressive learning path generation")
    print("="*70)


[INIT] Multi-Modal Interview System Initialized
[INIT] Total Questions: 5
[INIT] Question Types:
       - mcq: 2
       - voice: 1
       - coding: 1
       - subjective: 1


MULTI-MODAL QUESTION RECOMMENDATION DEMO

[DEMO 1] First Session - Baseline Assessment

[ENHANCED RECOMMENDATIONS] Multi-Modal Analysis
──────────────────────────────────────────────────────────────────────
[INFO] No history found - generating baseline assessment
[BASELINE] Creating diverse assessment across question types
{
  "user_id": "user_001",
  "session_type": "baseline_assessment",
  "domain": "java",
  "message": "Baseline assessment across all question types",
  "questions_recommended": [
    {
      "q_id": "q1",
      "score": 1.0,
      "type": "mcq",
      "reasoning": "Baseline mcq assessment for java, oop"
    },
    {
      "q_id": "q2",
      "score": 1.0,
      "type": "voice",
      "reasoning": "Baseline voice assessment for java, oop"
    },
    {
      "q_id": "q3",
      "score": 1.0,
    

In [4]:
"""
Multi-Modal Interview System - Web API Version
JSON-based input/output with separated data layer
"""

from typing import List, Dict, Optional, Tuple
from datetime import datetime
from dataclasses import dataclass, field, asdict
from collections import defaultdict
import json

# ==================== DATA STRUCTURES ====================

@dataclass
class AudioMetrics:
    """Detailed audio analysis from speech"""
    silence_ratio: float
    speech_rate_wpm: float
    pitch_variation: float
    clarity_score: float
    pacing_consistency: float
    expressiveness: float
    confidence_score: float
    transcription_confidence: str

@dataclass
class ContentMetrics:
    """Content quality assessment"""
    grammar_score: float
    relevance_score: float
    completeness_score: float
    overall_score: float
    missing_points: List[str]
    grammar_issues: List[str]

@dataclass
class UserProfile:
    """Enhanced User Profile with Multi-Modal Performance"""
    _id: str
    name: str
    email: str
    skills: List[str]
    skill_levels: Dict[str, int]
    years_experience: float
    education: str
    target_domains: List[str]
    target_companies: List[str]
    created_at: str
    last_active: str
    
    question_type_performance: Dict[str, Dict] = field(default_factory=dict)
    weaker_skills: List[str] = field(default_factory=list)
    stronger_skills: List[str] = field(default_factory=list)
    skill_metadata: Dict[str, Dict] = field(default_factory=dict)
    domain_specific_weaker_skills: Dict[str, List[str]] = field(default_factory=dict)
    domain_specific_stronger_skills: Dict[str, List[str]] = field(default_factory=dict)
    domain_specific_skill_metadata: Dict[str, Dict[str, Dict]] = field(default_factory=dict)
    avg_communication_score: float = 0.0
    communication_strengths: List[str] = field(default_factory=list)
    communication_weaknesses: List[str] = field(default_factory=list)

@dataclass
class Question:
    """Enhanced Question with Type and Difficulty Metadata"""
    _id: str
    text: str
    tags: List[str]
    domain: str
    difficulty: int
    estimated_time_sec: int
    question_type: str
    
    embedding: List[float] = field(default_factory=list)
    created_by: str = "editor"
    popularity: int = 0
    company_specific: List[str] = field(default_factory=list)
    last_updated: str = ""
    requires_audio_analysis: bool = False
    requires_code_execution: bool = False
    has_multiple_correct_answers: bool = False

@dataclass
class QuestionResponse:
    """Enhanced Response with Multi-Modal Assessment"""
    _id: str
    session_id: str
    user_id: str
    question_id: str
    question_type: str
    response_text: str
    
    audio_metrics: Optional[Dict] = None
    content_metrics: Optional[Dict] = None
    final_score: float = 0.0
    timestamp: str = ""
    time_taken_sec: int = 0
    skipped: bool = False
    confidence_rating: Optional[int] = None
    keyword_matches: List[str] = field(default_factory=list)
    domain: str = ""
    technical_score: float = 0.0
    communication_score: float = 0.0
    completeness_score: float = 0.0

# ==================== DATA LAYER (MOCK DATABASE) ====================

class DataStore:
    """Mock database - replace with actual MongoDB calls in production"""
    
    def __init__(self):
        self.questions = []
        self.users = {}
        self.responses = defaultdict(list)
        self.domain_responses = defaultdict(lambda: defaultdict(list))
        self.sessions = defaultdict(list)
    
    def add_questions(self, questions_json: List[Dict]):
        """Add questions from JSON"""
        self.questions = [self._dict_to_question(q) for q in questions_json]
        return {"status": "success", "count": len(self.questions)}
    
    def get_all_questions(self) -> List[Question]:
        """Get all questions"""
        return self.questions
    
    def get_question_by_id(self, question_id: str) -> Optional[Question]:
        """Get specific question"""
        for q in self.questions:
            if q._id == question_id:
                return q
        return None
    
    def add_user(self, user_json: Dict):
        """Add or update user"""
        user = self._dict_to_user(user_json)
        self.users[user._id] = user
        return {"status": "success", "user_id": user._id}
    
    def get_user(self, user_id: str) -> Optional[UserProfile]:
        """Get user profile"""
        return self.users.get(user_id)
    
    def add_response(self, response_json: Dict):
        """Add response"""
        response = self._dict_to_response(response_json)
        self.responses[response.user_id].append(response)
        if response.domain:
            self.domain_responses[response.user_id][response.domain].append(response)
        return {"status": "success", "response_id": response._id}
    
    def get_user_responses(self, user_id: str, domain: str = None) -> List[QuestionResponse]:
        """Get user responses"""
        if domain:
            return self.domain_responses[user_id].get(domain, [])
        return self.responses.get(user_id, [])
    
    def get_attempted_questions(self, user_id: str, domain: str = None) -> set:
        """Get attempted question IDs"""
        responses = self.get_user_responses(user_id, domain)
        return {r.question_id for r in responses}
    
    # Helper conversion methods
    def _dict_to_question(self, d: Dict) -> Question:
        return Question(**{k: v for k, v in d.items() if k in Question.__annotations__})
    
    def _dict_to_user(self, d: Dict) -> UserProfile:
        return UserProfile(**{k: v for k, v in d.items() if k in UserProfile.__annotations__})
    
    def _dict_to_response(self, d: Dict) -> QuestionResponse:
        return QuestionResponse(**{k: v for k, v in d.items() if k in QuestionResponse.__annotations__})

# Global data store instance
data_store = DataStore()

# ==================== MULTI-MODAL ANALYSIS ====================

def analyze_audio_performance(audio_metrics: Dict) -> Dict:
    """Analyze audio performance and identify strengths/weaknesses"""
    strengths = []
    weaknesses = []
    
    silence_ratio = audio_metrics.get('silence_ratio', 0)
    speech_rate = audio_metrics.get('speech_rate_wpm', 0)
    clarity = audio_metrics.get('clarity_score', 0)
    expressiveness = audio_metrics.get('expressiveness', 0)
    confidence = audio_metrics.get('confidence_score', 0)
    
    if silence_ratio < 15:
        strengths.append("confident_delivery")
    elif silence_ratio > 25:
        weaknesses.append("excessive_pauses")
    
    if 140 <= speech_rate <= 160:
        strengths.append("optimal_pace")
    elif speech_rate < 100:
        weaknesses.append("slow_speech")
    elif speech_rate > 180:
        weaknesses.append("rushed_speech")
    
    if clarity >= 95:
        strengths.append("clear_articulation")
    elif clarity < 85:
        weaknesses.append("unclear_speech")
    
    if expressiveness >= 60:
        strengths.append("expressive_communication")
    elif expressiveness < 40:
        weaknesses.append("monotone_delivery")
    
    if confidence >= 70:
        strengths.append("high_confidence")
    elif confidence < 50:
        weaknesses.append("low_confidence")
    
    return {
        'overall_score': confidence,
        'strengths': strengths,
        'weaknesses': weaknesses,
        'needs_improvement': len(weaknesses) > 2
    }

def analyze_content_quality(content_metrics: Dict) -> Dict:
    """Analyze content quality and completeness"""
    strengths = []
    issues = []
    
    grammar = content_metrics.get('grammar_score', 0)
    relevance = content_metrics.get('relevance_score', 0)
    completeness = content_metrics.get('completeness_score', 0)
    
    if grammar >= 95:
        strengths.append("excellent_grammar")
    elif grammar < 80:
        issues.extend(content_metrics.get('grammar_issues', []))
    
    if relevance >= 90:
        strengths.append("highly_relevant")
    elif relevance < 70:
        issues.append("off_topic_content")
    
    if completeness >= 80:
        strengths.append("comprehensive_answer")
    elif completeness < 60:
        missing = content_metrics.get('missing_points', [])[:3]
        if missing:
            issues.append(f"missed_points: {', '.join(missing)}")
    
    return {
        'overall_score': content_metrics.get('overall_score', 0),
        'strengths': strengths,
        'issues': issues,
        'needs_improvement': len(issues) > 1
    }

def calculate_multi_modal_score(response: QuestionResponse) -> Tuple[float, Dict]:
    """Calculate comprehensive score from multi-modal data"""
    scores = {}
    weights = {}
    
    if response.question_type == 'voice':
        if response.content_metrics:
            scores['technical'] = response.content_metrics['overall_score'] / 100
            weights['technical'] = 0.6
        
        if response.audio_metrics:
            scores['communication'] = response.audio_metrics['confidence_score'] / 100
            weights['communication'] = 0.4
    
    elif response.question_type == 'mcq':
        scores['accuracy'] = response.final_score
        weights['accuracy'] = 0.8
        
        if response.time_taken_sec > 0:
            time_factor = min(1.0, 60 / response.time_taken_sec)
            scores['efficiency'] = time_factor
            weights['efficiency'] = 0.2
    
    elif response.question_type == 'coding':
        if response.content_metrics:
            scores['correctness'] = response.content_metrics['overall_score'] / 100
            weights['correctness'] = 0.7
            scores['completeness'] = response.content_metrics['completeness_score'] / 100
            weights['completeness'] = 0.3
    
    elif response.question_type == 'subjective':
        if response.content_metrics:
            scores['content'] = response.content_metrics['relevance_score'] / 100
            weights['content'] = 0.5
            scores['completeness'] = response.content_metrics['completeness_score'] / 100
            weights['completeness'] = 0.3
            scores['grammar'] = response.content_metrics['grammar_score'] / 100
            weights['grammar'] = 0.2
    
    if scores and weights:
        total_weight = sum(weights.values())
        final_score = sum(scores[k] * weights[k] for k in scores) / total_weight
    else:
        final_score = response.final_score
    
    return final_score, scores

# ==================== SKILL ANALYSIS ====================

def analyze_user_skills_multimodal(user_id: str, domain: str = None) -> List[Dict]:
    """Enhanced skill analysis with question type performance"""
    responses = data_store.get_user_responses(user_id, domain)
    
    if not responses:
        return []
    
    skill_data = defaultdict(lambda: {
        'scores': [],
        'times': [],
        'skips': [],
        'confidences': [],
        'keyword_matches': [],
        'type_scores': defaultdict(list),
        'communication_scores': [],
        'content_issues': []
    })
    
    for response in responses:
        question = data_store.get_question_by_id(response.question_id)
        if not question:
            continue
        
        final_score, score_breakdown = calculate_multi_modal_score(response)
        
        for tag in question.tags:
            skill_data[tag]['scores'].append(final_score)
            skill_data[tag]['times'].append(response.time_taken_sec)
            skill_data[tag]['skips'].append(1 if response.skipped else 0)
            skill_data[tag]['type_scores'][response.question_type].append(final_score)
            
            if response.confidence_rating:
                skill_data[tag]['confidences'].append(response.confidence_rating)
            
            if response.keyword_matches:
                skill_data[tag]['keyword_matches'].append(len(response.keyword_matches))
            
            if response.audio_metrics:
                audio_analysis = analyze_audio_performance(response.audio_metrics)
                skill_data[tag]['communication_scores'].append(audio_analysis['overall_score'])
                if audio_analysis['weaknesses']:
                    skill_data[tag]['content_issues'].extend(audio_analysis['weaknesses'])
            
            if response.content_metrics:
                content_analysis = analyze_content_quality(response.content_metrics)
                if content_analysis['issues']:
                    skill_data[tag]['content_issues'].extend(content_analysis['issues'])
    
    analyses = []
    for skill, data in skill_data.items():
        if not data['scores']:
            continue
        
        avg_score = sum(data['scores']) / len(data['scores'])
        avg_time = sum(data['times']) / len(data['times']) if data['times'] else 0
        skip_rate = sum(data['skips']) / len(data['skips'])
        
        perf_by_type = {}
        for q_type, scores in data['type_scores'].items():
            if scores:
                perf_by_type[q_type] = sum(scores) / len(scores)
        
        avg_comm_score = None
        if data['communication_scores']:
            avg_comm_score = sum(data['communication_scores']) / len(data['communication_scores'])
        
        if avg_score < 0.5 or skip_rate > 0.3:
            category = 'weaker'
        elif avg_score > 0.75 and skip_rate < 0.1:
            category = 'stronger'
        else:
            category = 'moderate'
        
        unique_issues = list(set(data['content_issues']))[:3]
        
        analyses.append({
            'skill': skill,
            'avg_score': round(avg_score, 2),
            'total_questions': len(data['scores']),
            'avg_time_sec': round(avg_time, 1),
            'skip_rate': round(skip_rate, 2),
            'avg_confidence': round(sum(data['confidences']) / len(data['confidences']), 2) if data['confidences'] else None,
            'keyword_match_rate': round(sum(data['keyword_matches']) / len(data['keyword_matches']), 2) if data['keyword_matches'] else 0,
            'category': category,
            'performance_by_type': {k: round(v, 2) for k, v in perf_by_type.items()},
            'avg_communication_score': round(avg_comm_score, 1) if avg_comm_score else None,
            'content_issues': unique_issues
        })
    
    return sorted(analyses, key=lambda x: x['avg_score'])

# ==================== QUESTION TYPE SELECTION ====================

def select_optimal_question_type(skill_analysis: Dict, user_profile: UserProfile) -> str:
    """Select best question type based on user's performance patterns"""
    
    perf_by_type = skill_analysis.get('performance_by_type', {})
    category = skill_analysis.get('category', 'moderate')
    content_issues = skill_analysis.get('content_issues', [])
    
    if 'voice' in perf_by_type:
        voice_score = perf_by_type['voice']
        if voice_score < 0.6 and any('unclear' in issue or 'confidence' in issue for issue in content_issues):
            return 'voice'
    
    if 'mcq' in perf_by_type and 'subjective' in perf_by_type:
        mcq_score = perf_by_type['mcq']
        subj_score = perf_by_type.get('subjective', 0)
        if mcq_score > 0.8 and subj_score < 0.6:
            return 'subjective'
    
    if category == 'weaker':
        if not perf_by_type:
            return 'mcq'
        elif 'mcq' in perf_by_type and perf_by_type['mcq'] > 0.7:
            return 'subjective'
    
    if category == 'stronger':
        if 'coding' in perf_by_type:
            return 'coding'
        return 'voice'
    
    return 'subjective'

def score_questions_with_type_matching(questions: List[Question], 
                                       skill_analyses: List[Dict],
                                       user_profile: UserProfile,
                                       focus_type: str) -> List[Dict]:
    """Enhanced scoring with question type optimization"""
    scored = []
    skill_map = {a['skill']: a for a in skill_analyses}
    
    for question in questions:
        question_skills = [s for s in question.tags if s in skill_map]
        if not question_skills:
            continue
        
        avg_skill_score = sum(skill_map[s]['avg_score'] for s in question_skills) / len(question_skills)
        
        primary_skill = question_skills[0]
        optimal_type = select_optimal_question_type(skill_map[primary_skill], user_profile)
        
        type_match_bonus = 0.3 if question.question_type == optimal_type else 0
        
        if focus_type == 'weaker':
            priority = (1 - avg_skill_score) + type_match_bonus
            issues = skill_map[primary_skill].get('content_issues', [])
            issue_str = f" (addressing: {', '.join(issues[:2])})" if issues else ""
            reasoning = f"{question.question_type.upper()}: Focus on {', '.join(question_skills)} (current: {avg_skill_score:.0%}){issue_str}"
        else:
            priority = avg_skill_score + type_match_bonus
            reasoning = f"{question.question_type.upper()}: Validate {', '.join(question_skills)} mastery (strong: {avg_skill_score:.0%})"
        
        factors = ["skill_match", f"type_{question.question_type}"]
        if type_match_bonus > 0:
            factors.append("optimal_format")
        
        scored.append({
            'question': question,
            'score': priority,
            'reasoning': reasoning,
            'factors': factors
        })
    
    return scored

# ==================== RECOMMENDATION ENGINE ====================

def generate_recommendations(user_id: str, k: int = 10, chosen_domain: str = None) -> Dict:
    """Main recommendation engine - JSON input/output"""
    
    user = data_store.get_user(user_id)
    if not user:
        return {
            "status": "error",
            "message": "User not found"
        }
    
    skill_analyses = analyze_user_skills_multimodal(user_id, chosen_domain)
    
    if not skill_analyses:
        return generate_baseline_assessment(user_id, k, chosen_domain)
    
    weaker_skills = [a for a in skill_analyses if a['category'] == 'weaker']
    stronger_skills = [a for a in skill_analyses if a['category'] == 'stronger']
    
    target_domains = [chosen_domain] if chosen_domain else user.target_domains
    attempted = data_store.get_attempted_questions(user_id, chosen_domain)
    
    all_questions = data_store.get_all_questions()
    candidates = [q for q in all_questions if q.domain in target_domains and q._id not in attempted]
    
    weaker_scored = score_questions_with_type_matching(candidates, weaker_skills, user, 'weaker')
    stronger_scored = score_questions_with_type_matching(candidates, stronger_skills, user, 'stronger')
    
    k_weaker = int(k * 0.7)
    k_stronger = k - k_weaker
    
    weaker_ranked = sorted(weaker_scored, key=lambda x: x['score'], reverse=True)[:k_weaker]
    stronger_ranked = sorted(stronger_scored, key=lambda x: x['score'], reverse=True)[:k_stronger]
    
    recommendations = []
    
    for item in weaker_ranked:
        recommendations.append({
            'question_id': item['question']._id,
            'question_text': item['question'].text,
            'question_type': item['question'].question_type,
            'tags': item['question'].tags,
            'difficulty': item['question'].difficulty,
            'estimated_time_sec': item['question'].estimated_time_sec,
            'score': round(item['score'], 2),
            'reasoning': item['reasoning'],
            'priority_factors': item['factors']
        })
    
    for item in stronger_ranked:
        recommendations.append({
            'question_id': item['question']._id,
            'question_text': item['question'].text,
            'question_type': item['question'].question_type,
            'tags': item['question'].tags,
            'difficulty': item['question'].difficulty,
            'estimated_time_sec': item['question'].estimated_time_sec,
            'score': round(item['score'], 2),
            'reasoning': item['reasoning'],
            'priority_factors': item['factors']
        })
    
    return {
        "status": "success",
        "user_id": user_id,
        "session_type": "multi_modal_adaptive",
        "domain": chosen_domain,
        "skill_analyses": skill_analyses,
        "total_recommendations": len(recommendations),
        "questions_recommended": recommendations
    }

def generate_baseline_assessment(user_id: str, k: int, chosen_domain: str) -> Dict:
    """Initial assessment with diverse question types"""
    
    user = data_store.get_user(user_id)
    if not user:
        return {"status": "error", "message": "User not found"}
    
    target_domains = [chosen_domain] if chosen_domain else user.target_domains
    all_questions = data_store.get_all_questions()
    candidates = [q for q in all_questions if q.domain in target_domains]
    
    type_distribution = {'mcq': 0.3, 'subjective': 0.3, 'voice': 0.2, 'coding': 0.2}
    recommendations = []
    
    for q_type, ratio in type_distribution.items():
        type_questions = [q for q in candidates if q.question_type == q_type]
        count = int(k * ratio)
        
        for q in type_questions[:count]:
            recommendations.append({
                'question_id': q._id,
                'question_text': q.text,
                'question_type': q.question_type,
                'tags': q.tags,
                'difficulty': q.difficulty,
                'estimated_time_sec': q.estimated_time_sec,
                'score': 1.0,
                'reasoning': f"Baseline {q_type} assessment for {', '.join(q.tags)}",
                'priority_factors': ["baseline", "skill_coverage"]
            })
    
    return {
        "status": "success",
        "user_id": user_id,
        "session_type": "baseline_assessment",
        "domain": chosen_domain,
        "message": "Baseline assessment across all question types",
        "total_recommendations": len(recommendations),
        "questions_recommended": recommendations
    }

# ==================== API FUNCTIONS ====================

def setup_questions(questions_json: str) -> str:
    """Setup questions from JSON string"""
    try:
        questions = json.loads(questions_json)
        result = data_store.add_questions(questions)
        return json.dumps(result)
    except Exception as e:
        return json.dumps({"status": "error", "message": str(e)})

def setup_user(user_json: str) -> str:
    """Setup user from JSON string"""
    try:
        user_data = json.loads(user_json)
        result = data_store.add_user(user_data)
        return json.dumps(result)
    except Exception as e:
        return json.dumps({"status": "error", "message": str(e)})

def log_response(response_json: str) -> str:
    """Log response from JSON string"""
    try:
        response_data = json.loads(response_json)
        result = data_store.add_response(response_data)
        return json.dumps(result)
    except Exception as e:
        return json.dumps({"status": "error", "message": str(e)})

def get_recommendations(request_json: str) -> str:
    """Get recommendations from JSON request"""
    try:
        request = json.loads(request_json)
        user_id = request.get('user_id')
        k = request.get('k', 10)
        domain = request.get('domain')
        
        result = generate_recommendations(user_id, k, domain)
        return json.dumps(result, indent=2)
    except Exception as e:
        return json.dumps({"status": "error", "message": str(e)})

def get_skill_analysis(request_json: str) -> str:
    """Get detailed skill analysis"""
    try:
        request = json.loads(request_json)
        user_id = request.get('user_id')
        domain = request.get('domain')
        
        analyses = analyze_user_skills_multimodal(user_id, domain)
        return json.dumps({
            "status": "success",
            "user_id": user_id,
            "domain": domain,
            "skill_analyses": analyses
        }, indent=2)
    except Exception as e:
        return json.dumps({"status": "error", "message": str(e)})

# ==================== DEMO ====================

if __name__ == "__main__":
    print("="*80)
    print("MULTI-MODAL INTERVIEW SYSTEM - WEB API VERSION")
    print("="*80)
    
    # 1. Setup Questions
    print("\n[1] Setting up questions...")
    questions_data = json.dumps([
        {
            "_id": "q1",
            "text": "Explain Java inheritance",
            "tags": ["java", "oop"],
            "domain": "java",
            "difficulty": 2,
            "estimated_time_sec": 120,
            "question_type": "mcq"
        },
        {
            "_id": "q2",
            "text": "Explain polymorphism in detail",
            "tags": ["java", "oop"],
            "domain": "java",
            "difficulty": 3,
            "estimated_time_sec": 180,
            "question_type": "voice"
        },
        {
            "_id": "q3",
            "text": "Implement binary search",
            "tags": ["java", "algorithms"],
            "domain": "java",
            "difficulty": 3,
            "estimated_time_sec": 300,
            "question_type": "coding"
        }
    ])
    print(setup_questions(questions_data))
    
    # 2. Setup User
    print("\n[2] Setting up user...")
    user_data = json.dumps({
        "_id": "user_001",
        "name": "John Doe",
        "email": "john@example.com",
        "skills": ["java", "python", "oop", "algorithms"],
        "skill_levels": {},
        "years_experience": 2.0,
        "education": "B.Tech",
        "target_domains": ["java"],
        "target_companies": ["TechCorp"],
        "created_at": "2025-01-01T00:00:00Z",
        "last_active": "2025-01-15T00:00:00Z"
    })
    print(setup_user(user_data))
    
    # 3. Get Initial Recommendations
    print("\n[3] Getting initial recommendations...")
    rec_request = json.dumps({
        "user_id": "user_001",
        "k": 5,
        "domain": "java"
    })
    print(get_recommendations(rec_request))
    
    # 4. Log Response
    print("\n[4] Logging response...")
    response_data = json.dumps({
        "_id": "resp_001",
        "session_id": "session_001",
        "user_id": "user_001",
        "question_id": "q2",
        "question_type": "voice",
        "response_text": "Polymorphism allows objects...",
        "audio_metrics": {
            "silence_ratio": 28.5,
            "speech_rate_wpm": 95.3,
            "pitch_variation": 25.4,
            "clarity_score": 82.1,
            "pacing_consistency": 15.3,
            "expressiveness": 38.2,
            "confidence_score": 42.8,
            "transcription_confidence": "medium"
        },
        "content_metrics": {
            "grammar_score": 85.0,
            "relevance_score": 72.3,
            "completeness_score": 65.0,
            "overall_score": 74.1,
            "missing_points": ["key concept", "example"],
            "grammar_issues": ["Minor issues"]
        },
        "final_score": 0.68,
        "timestamp": "2025-01-15T10:00:00Z",
        "time_taken_sec": 195,
        "domain": "java"
    })
    print(log_response(response_data))
    
    # 5. Get Skill Analysis
    print("\n[5] Getting skill analysis...")
    analysis_request = json.dumps({
        "user_id": "user_001",
        "domain": "java"
    })
    print(get_skill_analysis(analysis_request))
    
    # 6. Get Updated Recommendations
    print("\n[6] Getting updated recommendations...")
    print(get_recommendations(rec_request))
    
    print("\n" + "="*80)
    print("DEMO COMPLETE")
    print("="*80)

MULTI-MODAL INTERVIEW SYSTEM - WEB API VERSION

[1] Setting up questions...
{"status": "success", "count": 3}

[2] Setting up user...
{"status": "success", "user_id": "user_001"}

[3] Getting initial recommendations...
{
  "status": "success",
  "user_id": "user_001",
  "session_type": "baseline_assessment",
  "domain": "java",
  "message": "Baseline assessment across all question types",
  "total_recommendations": 3,
  "questions_recommended": [
    {
      "question_id": "q1",
      "question_text": "Explain Java inheritance",
      "question_type": "mcq",
      "tags": [
        "java",
        "oop"
      ],
      "difficulty": 2,
      "estimated_time_sec": 120,
      "score": 1.0,
      "reasoning": "Baseline mcq assessment for java, oop",
      "priority_factors": [
        "baseline",
        "skill_coverage"
      ]
    },
    {
      "question_id": "q2",
      "question_text": "Explain polymorphism in detail",
      "question_type": "voice",
      "tags": [
        "java",
  

In [6]:
"""
Enhanced Personalized Interview System - Web App Version
JSON Input/Output with Data Management Functions
"""

from typing import List, Dict, Set, Optional, Tuple
from datetime import datetime
from dataclasses import dataclass, field, asdict
from collections import defaultdict
import json

# ==================== DATA STRUCTURES ====================

@dataclass
class UserProfile:
    """User Profile Data Structure"""
    _id: str
    name: str
    email: str
    skills: List[str]
    skill_levels: Dict[str, int]
    years_experience: float
    education: str
    target_domains: List[str]
    target_companies: List[str]
    created_at: str
    last_active: str
    weaker_skills: List[str] = field(default_factory=list)
    stronger_skills: List[str] = field(default_factory=list)
    skill_metadata: Dict[str, Dict] = field(default_factory=dict)
    domain_specific_weaker_skills: Dict[str, List[str]] = field(default_factory=dict)
    domain_specific_stronger_skills: Dict[str, List[str]] = field(default_factory=dict)
    domain_specific_skill_metadata: Dict[str, Dict[str, Dict]] = field(default_factory=dict)

@dataclass
class Question:
    """Question Bank Data Structure"""
    _id: str
    text: str
    tags: List[str]
    domain: str
    difficulty: int
    estimated_time_sec: int
    embedding: List[float] = field(default_factory=list)
    created_by: str = "editor"
    popularity: int = 0
    company_specific: List[str] = field(default_factory=list)
    last_updated: str = ""

@dataclass
class QuestionResponse:
    """Individual Question Response with Enhanced Metadata"""
    _id: str
    session_id: str
    user_id: str
    question_id: str
    response_text: str
    response_audio_features: Dict[str, float]
    content_metrics: Dict[str, float]
    final_score: float
    timestamp: str
    time_taken_sec: int = 0
    skipped: bool = False
    confidence_rating: Optional[int] = None
    keyword_matches: List[str] = field(default_factory=list)
    domain: str = ""

@dataclass
class Session:
    """User Session Data with Enhanced Tracking"""
    _id: str
    user_id: str
    context: Dict[str, str]
    questions: List[Dict[str, str]]
    created_at: str
    metrics_snapshot: Dict[str, float]
    skill_performance: Dict[str, List[float]] = field(default_factory=dict)
    weaker_skills_identified: List[str] = field(default_factory=list)
    stronger_skills_identified: List[str] = field(default_factory=list)
    total_time_sec: int = 0
    questions_skipped: int = 0
    domain: str = ""

@dataclass
class SkillAnalysis:
    """Detailed skill analysis from session data"""
    skill: str
    avg_score: float
    total_questions: int
    avg_time_sec: float
    skip_rate: float
    avg_confidence: Optional[float]
    keyword_match_rate: float
    category: str

# ==================== DATA MANAGEMENT FUNCTIONS ====================

def get_all_questions() -> List[Dict]:
    """Get all questions from 'database' (static data for now)"""
    questions = [
        # Java questions
        {
            "_id": "q_java_0001", "text": "Explain Java inheritance",
            "tags": ["java", "oop"], "domain": "java",
            "difficulty": 2, "estimated_time_sec": 100
        },
        {
            "_id": "q_java_0002", "text": "Java interfaces vs abstract classes",
            "tags": ["java", "oop"], "domain": "java",
            "difficulty": 3, "estimated_time_sec": 120
        },
        {
            "_id": "q_java_0003", "text": "Java threading concepts",
            "tags": ["java", "concurrency"], "domain": "java",
            "difficulty": 4, "estimated_time_sec": 150
        },
        {
            "_id": "q_java_0004", "text": "Exception handling in Java",
            "tags": ["java", "error-handling"], "domain": "java",
            "difficulty": 2, "estimated_time_sec": 80
        },
        # Python questions
        {
            "_id": "q_py_0001", "text": "Python decorators explained",
            "tags": ["python", "advanced"], "domain": "python",
            "difficulty": 3, "estimated_time_sec": 100
        },
        {
            "_id": "q_py_0002", "text": "List comprehensions in Python",
            "tags": ["python", "data-structures"], "domain": "python",
            "difficulty": 2, "estimated_time_sec": 70
        },
        {
            "_id": "q_py_0003", "text": "Python generators and iterators",
            "tags": ["python", "advanced"], "domain": "python",
            "difficulty": 4, "estimated_time_sec": 130
        },
        {
            "_id": "q_py_0004", "text": "Python async/await",
            "tags": ["python", "concurrency"], "domain": "python",
            "difficulty": 4, "estimated_time_sec": 140
        },
        # ML questions
        {
            "_id": "q_ml_0001", "text": "Explain TF-IDF",
            "tags": ["nlp", "information-retrieval"], "domain": "ml",
            "difficulty": 2, "estimated_time_sec": 90
        },
        {
            "_id": "q_ml_0002", "text": "Binary search algorithm",
            "tags": ["data-structures", "algorithms"], "domain": "ml",
            "difficulty": 1, "estimated_time_sec": 60
        },
        {
            "_id": "q_ml_0003", "text": "Backpropagation explained",
            "tags": ["deep-learning", "tensorflow"], "domain": "ml",
            "difficulty": 3, "estimated_time_sec": 120
        },
        {
            "_id": "q_ml_0004", "text": "Word embedding concepts",
            "tags": ["nlp", "deep-learning"], "domain": "ml",
            "difficulty": 2, "estimated_time_sec": 100
        },
        {
            "_id": "q_ml_0005", "text": "Neural network in Python",
            "tags": ["python", "deep-learning"], "domain": "ml",
            "difficulty": 3, "estimated_time_sec": 180
        },
        {
            "_id": "q_ml_0006", "text": "Data preprocessing techniques",
            "tags": ["python", "data-structures"], "domain": "ml",
            "difficulty": 2, "estimated_time_sec": 90
        },
    ]
    return questions

def get_user_profile(user_id: str) -> Optional[Dict]:
    """Get user profile from 'database'"""
    # Static data - in real app, fetch from MongoDB
    users = {
        "user_123": {
            "_id": "user_123",
            "name": "Pavan",
            "email": "pavan@example.com",
            "skills": ["python", "java", "nlp", "tensorflow", "data-structures", "oop"],
            "skill_levels": {},
            "years_experience": 2.5,
            "education": "B.Tech",
            "target_domains": ["java", "python", "ml"],
            "target_companies": ["CompanyA"],
            "created_at": "2025-09-01T12:00:00Z",
            "last_active": "2025-10-04T10:00:00Z",
            "weaker_skills": [],
            "stronger_skills": [],
            "skill_metadata": {},
            "domain_specific_weaker_skills": {},
            "domain_specific_stronger_skills": {},
            "domain_specific_skill_metadata": {}
        }
    }
    return users.get(user_id)

def get_user_sessions(user_id: str, domain: str = None) -> List[Dict]:
    """Get user sessions from 'database'"""
    # Static data - in real app, fetch from MongoDB
    # MODIFIED: Return empty for demonstration of first vs subsequent sessions
    sessions = {
        "user_123": {
            # Commenting out Java sessions to show first session flow
            # "java": [
            #     {
            #         "_id": "session_java_001",
            #         "user_id": "user_123",
            #         "domain": "java",
            #         "created_at": "2025-10-04T10:00:00Z",
            #         "total_time_sec": 300,
            #         "questions_skipped": 1
            #     }
            # ],
            "python": [
                {
                    "_id": "session_py_001",
                    "user_id": "user_123",
                    "domain": "python",
                    "created_at": "2025-10-04T11:00:00Z",
                    "total_time_sec": 280,
                    "questions_skipped": 0
                }
            ]
        }
    }
    
    user_sessions = sessions.get(user_id, {})
    if domain:
        return user_sessions.get(domain, [])
    
    # Return all sessions across domains
    all_sessions = []
    for domain_sessions in user_sessions.values():
        all_sessions.extend(domain_sessions)
    return all_sessions

def get_user_responses(user_id: str, domain: str = None, session_id: str = None) -> List[Dict]:
    """Get user responses from 'database'"""
    # Static data - in real app, fetch from MongoDB
    responses = {
        "user_123": {
            # Commenting out Java responses to show first session flow
            # "java": [
            #     {
            #         "_id": "resp_java_001",
            #         "session_id": "session_java_001",
            #         "user_id": "user_123",
            #         "question_id": "q_java_0001",
            #         "final_score": 0.40,
            #         "time_taken_sec": 110,
            #         "skipped": False,
            #         "confidence_rating": 2,
            #         "keyword_matches": ["inheritance", "extends"],
            #         "domain": "java",
            #         "timestamp": "2025-10-04T10:02:00Z"
            #     },
            #     {
            #         "_id": "resp_java_002",
            #         "session_id": "session_java_001",
            #         "user_id": "user_123",
            #         "question_id": "q_java_0002",
            #         "final_score": 0.0,
            #         "time_taken_sec": 5,
            #         "skipped": True,
            #         "confidence_rating": None,
            #         "keyword_matches": [],
            #         "domain": "java",
            #         "timestamp": "2025-10-04T10:04:00Z"
            #     },
            #     {
            #         "_id": "resp_java_003",
            #         "session_id": "session_java_001",
            #         "user_id": "user_123",
            #         "question_id": "q_java_0004",
            #         "final_score": 0.65,
            #         "time_taken_sec": 85,
            #         "skipped": False,
            #         "confidence_rating": 3,
            #         "keyword_matches": ["try", "catch", "finally", "throws"],
            #         "domain": "java",
            #         "timestamp": "2025-10-04T10:06:00Z"
            #     }
            # ],
            "python": [
                {
                    "_id": "resp_py_001",
                    "session_id": "session_py_001",
                    "user_id": "user_123",
                    "question_id": "q_py_0001",
                    "final_score": 0.80,
                    "time_taken_sec": 95,
                    "skipped": False,
                    "confidence_rating": 4,
                    "keyword_matches": ["decorator", "wrapper", "@", "function"],
                    "domain": "python",
                    "timestamp": "2025-10-04T11:02:00Z"
                },
                {
                    "_id": "resp_py_002",
                    "session_id": "session_py_001",
                    "user_id": "user_123",
                    "question_id": "q_py_0002",
                    "final_score": 0.88,
                    "time_taken_sec": 65,
                    "skipped": False,
                    "confidence_rating": 5,
                    "keyword_matches": ["comprehension", "list", "for", "if", "syntax"],
                    "domain": "python",
                    "timestamp": "2025-10-04T11:04:00Z"
                },
                {
                    "_id": "resp_py_003",
                    "session_id": "session_py_001",
                    "user_id": "user_123",
                    "question_id": "q_py_0003",
                    "final_score": 0.72,
                    "time_taken_sec": 120,
                    "skipped": False,
                    "confidence_rating": 4,
                    "keyword_matches": ["yield", "generator", "iterator", "lazy"],
                    "domain": "python",
                    "timestamp": "2025-10-04T11:07:00Z"
                }
            ]
        }
    }
    
    user_responses = responses.get(user_id, {})
    
    if domain and session_id:
        # Filter by both domain and session
        domain_responses = user_responses.get(domain, [])
        return [r for r in domain_responses if r["session_id"] == session_id]
    elif domain:
        return user_responses.get(domain, [])
    elif session_id:
        # Search across all domains
        all_responses = []
        for domain_responses in user_responses.values():
            all_responses.extend([r for r in domain_responses if r["session_id"] == session_id])
        return all_responses
    
    # Return all responses
    all_responses = []
    for domain_responses in user_responses.values():
        all_responses.extend(domain_responses)
    return all_responses

def save_session(session_data: Dict) -> Dict:
    """Save session to 'database'"""
    # In real app, save to MongoDB
    print(f"[SAVE] Session saved: {session_data['_id']}")
    return {"status": "success", "session_id": session_data["_id"]}

def save_response(response_data: Dict) -> Dict:
    """Save response to 'database'"""
    # In real app, save to MongoDB
    print(f"[SAVE] Response saved: {response_data['_id']}")
    return {"status": "success", "response_id": response_data["_id"]}

def update_user_profile(user_id: str, profile_updates: Dict) -> Dict:
    """Update user profile in 'database'"""
    # In real app, update MongoDB
    print(f"[UPDATE] User profile updated: {user_id}")
    return {"status": "success", "user_id": user_id}

# ==================== HELPER FUNCTIONS ====================

def dict_to_question(q_dict: Dict) -> Question:
    """Convert dictionary to Question object"""
    return Question(**q_dict)

def dict_to_response(r_dict: Dict) -> QuestionResponse:
    """Convert dictionary to QuestionResponse object"""
    return QuestionResponse(
        _id=r_dict["_id"],
        session_id=r_dict["session_id"],
        user_id=r_dict["user_id"],
        question_id=r_dict["question_id"],
        response_text=r_dict.get("response_text", ""),
        response_audio_features=r_dict.get("response_audio_features", {}),
        content_metrics=r_dict.get("content_metrics", {}),
        final_score=r_dict["final_score"],
        timestamp=r_dict["timestamp"],
        time_taken_sec=r_dict.get("time_taken_sec", 0),
        skipped=r_dict.get("skipped", False),
        confidence_rating=r_dict.get("confidence_rating"),
        keyword_matches=r_dict.get("keyword_matches", []),
        domain=r_dict.get("domain", "")
    )

# Add response_id property to QuestionResponse for easier access
@property
def response_id(self) -> str:
    return self._id

QuestionResponse.response_id = response_id

def filter_by_domain(questions: List[Question], target_domains: List[str]) -> List[Question]:
    """Filter questions by user's target domains"""
    return [q for q in questions if q.domain in target_domains]

def calculate_skill_overlap(question_tags: List[str], user_skills: List[str]) -> float:
    """Calculate skill overlap score"""
    if not question_tags:
        return 0.0
    intersection = len(set(question_tags) & set(user_skills))
    return intersection / len(question_tags)

def get_last_score(responses: List[QuestionResponse]) -> float:
    """Get the last response score"""
    if not responses:
        return 0.5
    sorted_responses = sorted(responses, key=lambda x: x.timestamp)
    return sorted_responses[-1].final_score

def calculate_difficulty_adjustment(last_score: float) -> int:
    """Adjust difficulty based on last score"""
    if last_score >= 0.7:
        return 1
    elif last_score < 0.4:
        return -1
    else:
        return 0

def get_attempted_questions(responses: List[QuestionResponse]) -> Set[str]:
    """Get set of question IDs already attempted"""
    return {response.question_id for response in responses}

def filter_candidate_questions(questions: List[Question], target_domains: List[str],
                               skills: List[str], target_difficulty: int,
                               attempted_questions: Set[str]) -> List[Question]:
    """Filter questions based on criteria"""
    candidates = []
    for question in questions:
        if question.domain not in target_domains:
            continue
        if not set(question.tags) & set(skills):
            continue
        if abs(question.difficulty - target_difficulty) > 1:
            continue
        if question._id in attempted_questions:
            continue
        candidates.append(question)
    return candidates

def get_question_by_id(questions: List[Question], question_id: str) -> Optional[Question]:
    """Retrieve question by ID"""
    for q in questions:
        if q._id == question_id:
            return q
    return None

# ==================== DIVERSE ASSESSMENT ====================

def create_diverse_assessment_set(questions: List[Question], skills: List[str], k: int) -> List[Question]:
    """Create diverse question set ensuring coverage of all skills"""
    skill_coverage = defaultdict(list)
    
    for question in questions:
        for tag in question.tags:
            if tag in skills:
                skill_coverage[tag].append(question)
    
    selected = []
    selected_ids = set()
    
    max_per_skill = max(1, k // len(skills))
    
    for skill in skills:
        skill_questions = skill_coverage.get(skill, [])
        for q in skill_questions[:max_per_skill]:
            if q._id not in selected_ids:
                selected.append(q)
                selected_ids.add(q._id)
                if len(selected) >= k:
                    break
        if len(selected) >= k:
            break
    
    if len(selected) < k:
        for q in questions:
            if q._id not in selected_ids:
                selected.append(q)
                selected_ids.add(q._id)
                if len(selected) >= k:
                    break
    
    return selected

# ==================== SKILL ANALYSIS ====================

def analyze_user_skills(questions: List[Question], responses: List[QuestionResponse]) -> List[SkillAnalysis]:
    """Analyze user skills from response history"""
    if not responses:
        return []
    
    skill_data = defaultdict(lambda: {
        'scores': [],
        'times': [],
        'skips': [],
        'confidences': [],
        'keyword_matches': []
    })
    
    for response in responses:
        question = get_question_by_id(questions, response.question_id)
        if not question:
            continue
        
        for tag in question.tags:
            skill_data[tag]['scores'].append(response.final_score)
            skill_data[tag]['times'].append(response.time_taken_sec)
            skill_data[tag]['skips'].append(1 if response.skipped else 0)
            if response.confidence_rating:
                skill_data[tag]['confidences'].append(response.confidence_rating)
            if response.keyword_matches:
                skill_data[tag]['keyword_matches'].append(len(response.keyword_matches))
    
    analyses = []
    for skill, data in skill_data.items():
        if not data['scores']:
            continue
        
        avg_score = sum(data['scores']) / len(data['scores'])
        avg_time = sum(data['times']) / len(data['times']) if data['times'] else 0
        skip_rate = sum(data['skips']) / len(data['skips'])
        avg_confidence = (
            sum(data['confidences']) / len(data['confidences'])
            if data['confidences'] else None
        )
        keyword_match_rate = (
            sum(data['keyword_matches']) / len(data['keyword_matches'])
            if data['keyword_matches'] else 0
        )
        
        if avg_score < 0.5 or skip_rate > 0.3:
            category = 'weaker'
        elif avg_score > 0.7 and skip_rate < 0.1:
            category = 'stronger'
        else:
            category = 'moderate'
        
        analyses.append(SkillAnalysis(
            skill=skill,
            avg_score=avg_score,
            total_questions=len(data['scores']),
            avg_time_sec=avg_time,
            skip_rate=skip_rate,
            avg_confidence=avg_confidence,
            keyword_match_rate=keyword_match_rate,
            category=category
        ))
    
    return sorted(analyses, key=lambda x: x.avg_score)

def score_profile_based_questions(questions: List[Question], skill_analyses: List[SkillAnalysis],
                                  focus_type: str) -> List[Tuple[Question, float, str]]:
    """Score questions based on user profile analysis"""
    scored = []
    skill_map = {a.skill: a for a in skill_analyses}
    
    for question in questions:
        question_skills = [s for s in question.tags if s in skill_map]
        
        if not question_skills:
            continue
        
        avg_skill_score = sum(
            skill_map[s].avg_score for s in question_skills
        ) / len(question_skills)
        
        avg_time = sum(
            skill_map[s].avg_time_sec for s in question_skills
        ) / len(question_skills)
        
        if focus_type == 'weaker':
            priority = (1 - avg_skill_score)
            time_factor = min(1.0, avg_time / 180)
            score = priority + (0.3 * time_factor)
            
            reasoning = (
                f"Focus on {', '.join(question_skills)} "
                f"(accuracy: {avg_skill_score:.0%}, "
                f"avg time: {avg_time:.0f}s) - needs improvement"
            )
        else:
            priority = avg_skill_score
            score = priority
            
            reasoning = (
                f"Validate {', '.join(question_skills)} "
                f"(strong performance: {avg_skill_score:.0%}) - maintain mastery"
            )
        
        scored.append((question, score, reasoning))
    
    return scored

# ==================== FIRST SESSION FLOW ====================

def first_session_recommendations(questions: List[Question], user_skills: List[str],
                                 target_domains: List[str], k: int) -> Dict:
    """First Session: Diverse question set to assess all skills"""
    print(f"\n[PHASE 1] First Session - Comprehensive Assessment")
    print(f"  → No prior session history found for this domain")
    print(f"  → Creating diverse assessment covering all skills: {user_skills}")
    
    domain_questions = filter_by_domain(questions, target_domains)
    print(f"  → Available questions in domain: {len(domain_questions)}")
    
    diverse_questions = create_diverse_assessment_set(domain_questions, user_skills, k)
    
    scored_questions = []
    for question in diverse_questions:
        skill_overlap = calculate_skill_overlap(question.tags, user_skills)
        difficulty_factor = 1.0
        if question.difficulty == 1 or question.difficulty == 5:
            difficulty_factor = 0.9
        
        score = skill_overlap * difficulty_factor
        scored_questions.append((question, score))
    
    ranked = sorted(scored_questions, key=lambda x: x[1], reverse=True)
    
    recommendations = [
        {
            "q_id": q._id,
            "score": round(score, 2),
            "reasoning": f"Initial assessment for {', '.join(q.tags)}",
            "question_text": q.text,
            "tags": q.tags,
            "difficulty": q.difficulty
        }
        for q, score in ranked[:k]
    ]
    
    print(f"  → Generated {len(recommendations)} diverse questions for initial assessment")
    
    return {
        "session_type": "cold_start_first_session",
        "phase": "data_collection",
        "domain": target_domains[0] if target_domains else "general",
        "message": "First session: Comprehensive skill assessment",
        "questions_recommended": recommendations,
        "session_history": {
            "total_sessions": 0,
            "total_responses": 0,
            "sessions_used": [],
            "responses_used": []
        }
    }

# ==================== PROFILE-BASED FLOW ====================

def profile_based_recommendations(questions: List[Question], responses: List[QuestionResponse],
                                 user_skills: List[str], target_domains: List[str], k: int,
                                 sessions: List[Dict]) -> Dict:
    """Subsequent Sessions: Use built profile for targeted recommendations"""
    print(f"\n[PHASE 2] Profile-Based Recommendations")
    print(f"  → Found {len(sessions)} previous session(s) for this domain")
    print(f"  → Analyzing {len(responses)} response(s) to build skill profile")
    
    # Show responses being analyzed
    print(f"\n  Responses used for analysis:")
    for i, resp in enumerate(responses[:5], 1):  # Show first 5
        print(f"    {i}. Question: {resp.question_id}, Score: {resp.final_score:.2f}, Skipped: {resp.skipped}")
    if len(responses) > 5:
        print(f"    ... and {len(responses) - 5} more responses")
    
    skill_analyses = analyze_user_skills(questions, responses)
    
    if not skill_analyses:
        print(f"  ⚠ No skill analysis available, falling back to first session")
        return first_session_recommendations(questions, user_skills, target_domains, k)
    
    weaker_skills = [a.skill for a in skill_analyses if a.category == 'weaker']
    stronger_skills = [a.skill for a in skill_analyses if a.category == 'stronger']
    
    print(f"\n  Skill Analysis Results:")
    print(f"    → Weaker skills (need focus): {weaker_skills}")
    print(f"    → Stronger skills (maintain): {stronger_skills}")
    
    last_score = get_last_score(responses)
    difficulty_adjustment = calculate_difficulty_adjustment(last_score)
    target_difficulty = max(1, min(5, 2 + difficulty_adjustment))
    
    print(f"\n  Difficulty Adaptation:")
    print(f"    → Last score: {last_score:.2f}")
    print(f"    → Target difficulty: {target_difficulty}")
    
    attempted = get_attempted_questions(responses)
    print(f"    → Previously attempted: {len(attempted)} questions")
    
    weaker_candidates = filter_candidate_questions(
        questions, target_domains, weaker_skills, target_difficulty, attempted
    )
    
    stronger_candidates = filter_candidate_questions(
        questions, target_domains, stronger_skills, target_difficulty, attempted
    )
    
    print(f"\n  Candidate Questions:")
    print(f"    → For weaker skills: {len(weaker_candidates)} available")
    print(f"    → For stronger skills: {len(stronger_candidates)} available")
    
    weaker_scored = score_profile_based_questions(weaker_candidates, skill_analyses, 'weaker')
    stronger_scored = score_profile_based_questions(stronger_candidates, skill_analyses, 'stronger')
    
    k_weaker = int(k * 0.7)
    k_stronger = k - k_weaker
    
    weaker_ranked = sorted(weaker_scored, key=lambda x: x[1], reverse=True)[:k_weaker]
    stronger_ranked = sorted(stronger_scored, key=lambda x: x[1], reverse=True)[:k_stronger]
    
    recommendations = []
    
    for q, score, reasoning in weaker_ranked:
        recommendations.append({
            "q_id": q._id,
            "score": round(score, 2),
            "reasoning": reasoning,
            "question_text": q.text,
            "tags": q.tags,
            "difficulty": q.difficulty,
            "focus_area": "weaker_skill"
        })
    
    for q, score, reasoning in stronger_ranked:
        recommendations.append({
            "q_id": q._id,
            "score": round(score, 2),
            "reasoning": reasoning,
            "question_text": q.text,
            "tags": q.tags,
            "difficulty": q.difficulty,
            "focus_area": "stronger_skill"
        })
    
    print(f"\n  → Generated {len(recommendations)} recommendations ({k_weaker} for improvement, {k_stronger} for validation)")
    
    # Prepare session history details
    session_history = {
        "total_sessions": len(sessions),
        "total_responses": len(responses),
        "sessions_used": [
            {
                "session_id": s["_id"],
                "created_at": s["created_at"],
                "questions_skipped": s.get("questions_skipped", 0)
            } for s in sessions
        ],
        "responses_used": [
            {
                "response_id": r.response_id,
                "question_id": r.question_id,
                "score": r.final_score,
                "time_taken_sec": r.time_taken_sec,
                "skipped": r.skipped
            } for r in responses
        ]
    }
    
    return {
        "session_type": "cold_start_profile_based",
        "phase": "adaptive_learning",
        "domain": target_domains[0] if target_domains else "general",
        "weaker_skills_targeted": weaker_skills,
        "stronger_skills_validated": stronger_skills,
        "skill_analyses": [
            {
                "skill": a.skill,
                "category": a.category,
                "avg_score": round(a.avg_score, 2),
                "total_questions": a.total_questions,
                "avg_time_sec": round(a.avg_time_sec, 1)
            }
            for a in skill_analyses
        ],
        "questions_recommended": recommendations,
        "session_history": session_history
    }

# ==================== MAIN API FUNCTION ====================

def recommend_questions_api(request_json: Dict) -> Dict:
    """
    Main API function - accepts JSON input and returns JSON output
    
    Input JSON format:
    {
        "user_id": "user_123",
        "domain": "python",
        "k": 3
    }
    
    Output JSON format:
    {
        "user_id": "user_123",
        "session_type": "...",
        "phase": "...",
        "domain": "python",
        "questions_recommended": [...]
    }
    """
    try:
        # Extract input parameters
        user_id = request_json.get("user_id")
        chosen_domain = request_json.get("domain")
        k = request_json.get("k", 10)
        
        if not user_id:
            return {"error": "user_id is required"}
        
        # Get data from 'database'
        user_profile_dict = get_user_profile(user_id)
        if not user_profile_dict:
            return {"error": f"User {user_id} not found"}
        
        # Convert to objects
        questions_dicts = get_all_questions()
        questions = [dict_to_question(q) for q in questions_dicts]
        
        # Get sessions and responses
        sessions = get_user_sessions(user_id, chosen_domain)
        responses_dicts = get_user_responses(user_id, chosen_domain)
        responses = [dict_to_response(r) for r in responses_dicts]
        
        # Determine if first session
        is_first_session = len(sessions) == 0
        
        # Get recommendations
        target_domains = [chosen_domain] if chosen_domain else user_profile_dict["target_domains"]
        
        if is_first_session:
            result = first_session_recommendations(
                questions, 
                user_profile_dict["skills"],
                target_domains,
                k
            )
        else:
            result = profile_based_recommendations(
                questions,
                responses,
                user_profile_dict["skills"],
                target_domains,
                k,
                sessions
            )
        
        # Add user_id to result
        result["user_id"] = user_id
        
        return result
        
    except Exception as e:
        return {"error": str(e)}

# ==================== EXAMPLE USAGE ====================

if __name__ == "__main__":
    print("=" * 80)
    print("INTERVIEW RECOMMENDATION SYSTEM - WEB APP DEMO")
    print("=" * 80)
    
    # Example 1: First session for Java (no prior history)
    print("\n" + "=" * 80)
    print("Example 1: Java Domain - FIRST SESSION (No Prior History)")
    print("=" * 80)
    
    request1 = {
        "user_id": "user_123",
        "domain": "java",
        "k": 3
    }
    
    response1 = recommend_questions_api(request1)
    print(json.dumps(response1, indent=2))
    
    # Example 2: Python - Profile-based (has history)
    print("\n" + "=" * 80)
    print("Example 2: Python Domain - PROFILE-BASED (Has Session History)")
    print("=" * 80)
    
    request2 = {
        "user_id": "user_123",
        "domain": "python",
        "k": 3
    }
    
    response2 = recommend_questions_api(request2)
    print(json.dumps(response2, indent=2))
    
    # Example 3: ML - First session for new domain
    print("\n" + "=" * 80)
    print("Example 3: ML Domain - FIRST SESSION (New Domain)")
    print("=" * 80)
    
    request3 = {
        "user_id": "user_123",
        "domain": "ml",
        "k": 4
    }
    
    response3 = recommend_questions_api(request3)
    print(json.dumps(response3, indent=2))
    
    print("\n" + "=" * 80)
    print("DEMO COMPLETED - All Examples Show Correct Session Types!")
    print("=" * 80)
    print("\nKey Features Demonstrated:")
    print("  ✓ First Session: Diverse skill assessment with no prior history")
    print("  ✓ Profile-Based: Adaptive recommendations using response history")
    print("  ✓ Session History: Shows which sessions/responses were analyzed")
    print("  ✓ Question Details: Includes text, tags, difficulty, focus area")
    print("  ✓ JSON Input/Output: Ready for web API integration")

INTERVIEW RECOMMENDATION SYSTEM - WEB APP DEMO

Example 1: Java Domain - FIRST SESSION (No Prior History)

[PHASE 1] First Session - Comprehensive Assessment
  → No prior session history found for this domain
  → Creating diverse assessment covering all skills: ['python', 'java', 'nlp', 'tensorflow', 'data-structures', 'oop']
  → Available questions in domain: 4
  → Generated 3 diverse questions for initial assessment
{
  "session_type": "cold_start_first_session",
  "phase": "data_collection",
  "domain": "java",
  "message": "First session: Comprehensive skill assessment",
  "questions_recommended": [
    {
      "q_id": "q_java_0001",
      "score": 1.0,
      "reasoning": "Initial assessment for java, oop",
      "question_text": "Explain Java inheritance",
      "tags": [
        "java",
        "oop"
      ],
      "difficulty": 2
    },
    {
      "q_id": "q_java_0002",
      "score": 1.0,
      "reasoning": "Initial assessment for java, oop",
      "question_text": "Java inter